# Ketamine 

In [1]:
%load_ext autoreload
%autoreload 2
%matplotlib tk
import numpy as np
import matplotlib.pyplot as plt
from Functions.generate_OU import get_mixed_OU_signals
from Functions.time_frequency import spectrogram
import pywt
import cv2
from sklearn.decomposition import NMF

C:\Program Files\WindowsApps\PythonSoftwareFoundation.Python.3.11_3.11.2544.0_x64__qbz5n2kfra8p0\Lib\tkinter\__init__.py:1967: UserWarning: This figure was using a layout engine that is incompatible with subplots_adjust and/or tight_layout; not calling subplots_adjust.
  return self.func(*args)


In [215]:
# --- Parameters
T = 20 # desired signal duration (s)
dt = 0.001
fs = 1 / dt

lbda_list = [1, 2, 1]
omega_list = [2*np.pi*1, 2*np.pi*10, 2*np.pi*30]
sigma_list = [3, 2, 2]
factor_list = [1, 1, 1]

# --- Generate EEG data
t, y = get_mixed_OU_signals(T, dt, lbda_list, omega_list, sigma_list, factor_list)

# --- compute spectrogram
f_spectro, t_spectro, spectro = spectrogram(y, fs, nfft_factor=2)
# mask_f = f_spectro >= 20
# f_spectro = f_spectro[mask_f]
# spectro = spectro[mask_f, :]

# --- Display
fig, axes = plt.subplots(3,  constrained_layout = True)
axes[0].plot(t, y)
axes[0].set_title('Simulated EEG signal')
axes[1].pcolormesh(t_spectro, f_spectro, np.log2(spectro + 1e-11), shading = 'nearest', cmap = 'jet')
axes[1].set_title('Spectrogram')
axes[2].plot(f_spectro, np.log2(np.median(spectro, axis = 1)))
axes[2].set_title('PSD')
axes[1].sharex(axes[0])

plt.show()

In [216]:
M = spectro.copy()
mask_f = f_spectro >= 20
f_M = f_spectro[mask_f]
M = M[mask_f, :]

fig, axis = plt.subplots(1)
axis.pcolormesh(t_spectro, f_M, np.log2(M + 1e-11), shading = 'nearest', cmap = 'jet')

In [ ]:
coeffs = pywt.dwt2(M, 'sym4')
cA, (cH, cV, cD) = coeffs

n_f, n_t = np.shape(cA) # shape of 2D DWT coefficients
t_wav = np.linspace(t_spectro[0], t_spectro[-1], n_t)
f_wav = np.linspace(f_spectro[0], f_spectro[-1], n_f)


fig, axes = plt.subplots(2, 3, sharex = True, constrained_layout = True)
axes[0, 0].plot(t, y)
axes[0, 0].set_title('Simulated EEG signal')
axes[1, 0].pcolormesh(t_spectro, f_M, M, shading = 'nearest', cmap = 'jet')
axes[1, 0].set_title('Spectrogram')

axes[0, 1].pcolormesh(t_wav, f_wav, cA, shading = 'nearest', cmap = 'jet')
axes[0, 1].set_title('cA coeff')
axes[1, 1].pcolormesh(t_wav, f_wav, cH, shading = 'nearest', cmap = 'jet')
axes[1, 1].set_title('cH coeff')

axes[0, 2].pcolormesh(t_wav, f_wav, cV, shading = 'nearest', cmap = 'jet')
axes[0, 2].set_title('cV coeff')
axes[1, 2].pcolormesh(t_wav, f_wav, cD, shading = 'nearest', cmap = 'jet')
axes[1, 2].set_title('cD coeff')

plt.show()

In [21]:
# Flatten and collect the 2D matrices into a dictionary for easy iteration
matrices = {
    'Spectrogram (M)': M,
    'cA coeff': cA,
    'cH coeff': cH,
    'cV coeff': cV,
    'cD coeff': cD
}

# Create a figure with 1 row and 5 subplots
fig_dist, axes_dist = plt.subplots(1, 5, figsize=(18, 4), constrained_layout=True)

for ax, (name, matrix) in zip(axes_dist, matrices.items()):
    # Flatten the 2D array to 1D to plot its distribution
    ax.hist(matrix.ravel(), bins=50, color='skyblue', edgecolor='black', alpha=0.7)
    ax.set_title(name)
    ax.set_xlabel('Value')
    ax.set_ylabel('Frequency')
    ax.grid(True, linestyle='--', alpha=0.6)

plt.show()

In [36]:
import numpy as np
import pywt
import matplotlib.pyplot as plt
from skimage.exposure import match_histograms

# =====================================================================
# 1. DEFINE PROCESS FUNCTIONS
# =====================================================================
def match_outliers_to_baseline(coeff, quantile=0.95):
    """
    Splits wavelet coefficients at a given quantile, matches the above-threshold
    outliers to the distribution of below-threshold values, and restores signs.
    Used for cH, cV, cD.
    """
    signs = np.sign(coeff)
    abs_coeff = np.abs(coeff)
    
    threshold = np.quantile(abs_coeff, quantile)
    baseline_mask = abs_coeff <= threshold
    outlier_mask = ~baseline_mask
    
    if not np.any(outlier_mask) or threshold == 0:
        return coeff, abs_coeff, abs_coeff
    
    baseline_values = abs_coeff[baseline_mask]
    outlier_values = abs_coeff[outlier_mask]
    
    matched_outliers = match_histograms(outlier_values, baseline_values)
    
    abs_coeff_filtered = abs_coeff.copy()
    abs_coeff_filtered[outlier_mask] = matched_outliers
    
    filtered_coeff = abs_coeff_filtered * signs
    return filtered_coeff, abs_coeff, abs_coeff_filtered

def match_cA_outliers_to_baseline(cA_coeff, quantile=0.95):
    """
    Specifically for cA (fully positive baseline energy). 
    Matches above-threshold cA values directly to below-threshold cA values.
    """
    threshold = np.quantile(cA_coeff, quantile)
    baseline_mask = cA_coeff <= threshold
    outlier_mask = ~baseline_mask
    
    if not np.any(outlier_mask):
        return cA_coeff, cA_coeff, cA_coeff
        
    baseline_values = cA_coeff[baseline_mask]
    outlier_values = cA_coeff[outlier_mask]
    
    matched_outliers = match_histograms(outlier_values, baseline_values)
    
    cA_filtered = cA_coeff.copy()
    cA_filtered[outlier_mask] = matched_outliers
    
    return cA_filtered, cA_coeff, cA_filtered

# =====================================================================
# 2. WAVELET DECOMPOSITION & SYMMETRIC FILTERING
# =====================================================================
# Decompose the spectrogram M
coeffs = pywt.dwt2(M, 'sym4')
cA, (cH, cV, cD) = coeffs

q = 0.8
# Apply the filter to ALL coefficients, including the cA baseline
cA_filt, cA_orig, cA_abs_filt = match_cA_outliers_to_baseline(cA, quantile=q)
cH_filt, cH_abs, cH_abs_filt = match_outliers_to_baseline(cH, quantile=q)
cV_filt, cV_abs, cV_abs_filt = match_outliers_to_baseline(cV, quantile=q)
cD_filt, cD_abs, cD_abs_filt = match_outliers_to_baseline(cD, quantile=q)

# Reconstruct the cleaned spectrogram
coeffs_filtered = cA_filt, (cH_filt, cV_filt, cD_filt)
M_clean = pywt.idwt2(coeffs_filtered, 'db1')

# Trim dimension mismatches if necessary
if M_clean.shape != M.shape:
    M_clean = M_clean[:M.shape[0], :M.shape[1]]

# =====================================================================
# 3. VISUALIZATION (Adding cA to the distribution monitoring)
# =====================================================================
fig = plt.figure(figsize=(15, 11), constrained_layout=True)
gs = fig.add_gridspec(2, 4)

# --- Row 1: Spectrogram Comparisons & Info ---
ax_orig_spec = fig.add_subplot(gs[0, 0:2])
im0 = ax_orig_spec.pcolormesh(t_spectro, f_M, M, shading='nearest', cmap='jet', vmin = 0.001, vmax = 0.12)
ax_orig_spec.set_title("Original Spectrogram (M)")
ax_orig_spec.set_ylabel("Frequency")
ax_orig_spec.set_xlabel("Time")
fig.colorbar(im0, ax=ax_orig_spec, label="Intensity")

ax_filt_spec = fig.add_subplot(gs[0, 2:4])
im1 = ax_filt_spec.pcolormesh(t_spectro, f_M, M_clean, shading='nearest', cmap='jet', vmin = 0.001, vmax = 0.12)
ax_filt_spec.set_title("Filtered Spectrogram (M_clean) - cA Modified")
ax_filt_spec.set_xlabel("Time")
fig.colorbar(im1, ax=ax_filt_spec, label="Intensity")

# --- Row 2: Absolute Coefficients Distribution Comparison ---
# Distribution of cA (Approximation)
ax_cA_dist = fig.add_subplot(gs[1, 0])
ax_cA_dist.hist(cA_orig.ravel(), bins=50, alpha=0.5, label='Original cA', color='crimson')
ax_cA_dist.hist(cA_abs_filt.ravel(), bins=50, alpha=0.7, label='Matched cA', color='mediumseagreen')
ax_cA_dist.set_title("cA Distribution")
ax_cA_dist.set_xlabel("Value")
ax_cA_dist.set_ylabel("Count")
ax_cA_dist.legend()
ax_cA_dist.grid(True, linestyle='--', alpha=0.5)

# Distribution of cH
ax_cH_dist = fig.add_subplot(gs[1, 1])
ax_cH_dist.hist(cH_abs.ravel(), bins=50, alpha=0.5, label='Original |cH|', color='crimson')
ax_cH_dist.hist(cH_abs_filt.ravel(), bins=50, alpha=0.7, label='Matched |cH|', color='mediumseagreen')
ax_cH_dist.set_title("cH Absolute Distribution")
ax_cH_dist.set_xlabel("Value")
ax_cH_dist.legend()
ax_cH_dist.grid(True, linestyle='--', alpha=0.5)

# Distribution of cV
ax_cV_dist = fig.add_subplot(gs[1, 2])
ax_cV_dist.hist(cV_abs.ravel(), bins=50, alpha=0.5, label='Original |cV|', color='crimson')
ax_cV_dist.hist(cV_abs_filt.ravel(), bins=50, alpha=0.7, label='Matched |cV|', color='mediumseagreen')
ax_cV_dist.set_title("cV Absolute Distribution")
ax_cV_dist.set_xlabel("Value")
ax_cV_dist.legend()
ax_cV_dist.grid(True, linestyle='--', alpha=0.5)

# Distribution of cD
ax_cD_dist = fig.add_subplot(gs[1, 3])
ax_cD_dist.hist(cD_abs.ravel(), bins=50, alpha=0.5, label='Original |cD|', color='crimson')
ax_cD_dist.hist(cD_abs_filt.ravel(), bins=50, alpha=0.7, label='Matched |cD|', color='mediumseagreen')
ax_cD_dist.set_title("cD Absolute Distribution")
ax_cD_dist.set_xlabel("Value")
ax_cD_dist.legend()
ax_cD_dist.grid(True, linestyle='--', alpha=0.5)

plt.show()

In [38]:
import numpy as np
import pywt
import matplotlib.pyplot as plt
from skimage.exposure import match_histograms

# Use a smooth wavelet family instead of the blocky 'db1'
WAVELET = 'sym4'  # 'sym4' or 'db4' work beautifully for keeping natural textures
LEVEL = 3         # Multi-level decomposition allows deeper baseline isolation

# =====================================================================
# 1. DEFINE SMOOTH PROCESS FUNCTIONS
# =====================================================================
def match_outliers_to_baseline(coeff, quantile=0.8):
    """ Matches high-intensity detail outliers to baseline """
    signs = np.sign(coeff)
    abs_coeff = np.abs(coeff)
    
    threshold = np.quantile(abs_coeff, quantile)
    baseline_mask = abs_coeff <= threshold
    outlier_mask = ~baseline_mask
    
    if not np.any(outlier_mask) or threshold == 0:
        return coeff
    
    baseline_values = abs_coeff[baseline_mask]
    outlier_values = abs_coeff[outlier_mask]
    
    matched_outliers = match_histograms(outlier_values, baseline_values)
    
    abs_coeff_filtered = abs_coeff.copy()
    abs_coeff_filtered[outlier_mask] = matched_outliers
    
    return abs_coeff_filtered * signs

def match_cA_outliers_to_baseline_gentle(cA_coeff, quantile=0.96):
    """ 
    Gently matches ONLY the extreme peaks of cA (e.g. top 4%), 
    leaving 96% of the smooth baseline background untouched.
    """
    threshold = np.quantile(cA_coeff, quantile)
    baseline_mask = cA_coeff <= threshold
    outlier_mask = ~baseline_mask
    
    if not np.any(outlier_mask):
        return cA_coeff
        
    baseline_values = cA_coeff[baseline_mask]
    outlier_values = cA_coeff[outlier_mask]
    
    matched_outliers = match_histograms(outlier_values, baseline_values)
    
    cA_filtered = cA_coeff.copy()
    cA_filtered[outlier_mask] = matched_outliers
    
    return cA_filtered

# =====================================================================
# 2. MULTI-LEVEL DECOMPOSITION & FILTERING
# =====================================================================
# Multi-level decomposition: coeffs is a list [cAn, (cHn, cVn, cDn), ..., (cH1, cV1, cD1)]
coeffs = pywt.wavedec2(M, WAVELET, level=LEVEL)

cA_n = coeffs[0]
details = coeffs[1:]

# 1. Apply very gentle matching to the ultra-low-frequency approximation
# We only target the top 4% (0.96) of baseline energy peaks
cA_n_filt = match_cA_outliers_to_baseline_gentle(cA_n, quantile=0.9)

# 2. Apply standard matching to the detail coefficients at all levels
details_filt = []
for level_details in details:
    cH, cV, cD = level_details
    # Filter details (we can keep a 0.8 quantile here as details are noise/edges)
    cH_f = match_outliers_to_baseline(cH, quantile=0.8)
    cV_f = match_outliers_to_baseline(cV, quantile=0.8)
    cD_f = match_outliers_to_baseline(cD, quantile=0.8)
    details_filt.append((cH_f, cV_f, cD_f))

# Reassemble the coefficients list
coeffs_filtered = [cA_n_filt] + details_filt

# =====================================================================
# 3. RECONSTRUCTION & PLOT
# =====================================================================
# Reconstruct using the smooth wavelets
M_smooth_clean = pywt.waverec2(coeffs_filtered, WAVELET)

# Adjust dimensions to match original M (due to multi-level padding)
if M_smooth_clean.shape != M.shape:
    M_smooth_clean = M_smooth_clean[:M.shape[0], :M.shape[1]]

# Plot comparisons
fig, axes = plt.subplots(1, 2, figsize=(14, 5), constrained_layout=True)

im0 = axes[0].pcolormesh(t_spectro, f_M, M, shading='nearest', cmap='jet', vmin = 0.001, vmax = 0.12)
axes[0].set_title("Original Spectrogram")
fig.colorbar(im0, ax=axes[0])

im1 = axes[1].pcolormesh(t_spectro, f_M, M_smooth_clean, shading='nearest', cmap='jet', vmin = 0.001, vmax = 0.12)
axes[1].set_title(f"Smooth Filtered Spectrogram ({WAVELET}, Level {LEVEL})")
fig.colorbar(im1, ax=axes[1])

plt.show()

In [39]:
fig, axis = plt.subplots(1)
im1 = axis.pcolormesh(t_spectro, f_M, M - M_smooth_clean, shading='nearest', cmap='jet')
axis.set_title(f"Smooth Filtered Spectrogram ({WAVELET}, Level {LEVEL})")
fig.colorbar(im1, ax=axis)

plt.show()

In [40]:
import numpy as np
import pywt
from skimage.exposure import match_histograms

def match_outliers_with_smooth_transition(coeff, quantile=0.8, transition_width=0.1):
    """
    Blends matched outliers smoothly with the baseline over a transition zone
    around the threshold, keeping ordering and eliminating boundary artifacts.
    """
    signs = np.sign(coeff)
    abs_coeff = np.abs(coeff)
    
    # 1. Establish the base threshold
    threshold = np.quantile(abs_coeff, quantile)
    
    if threshold == 0:
        return coeff
    
    # 2. Extract strictly below-threshold baseline & above-threshold outliers
    baseline_mask = abs_coeff <= threshold
    outlier_mask = ~baseline_mask
    
    if not np.any(outlier_mask):
        return coeff
    
    baseline_values = abs_coeff[baseline_mask]
    outlier_values = abs_coeff[outlier_mask]
    
    # 3. Match the outliers to the baseline distribution
    matched_outliers = match_histograms(outlier_values, baseline_values)
    
    # Prepare our fully matched array
    matched_full = abs_coeff.copy()
    matched_full[outlier_mask] = matched_outliers
    
    # 4. Create a smooth blending weight (alpha mask)
    # alpha = 0 means keep original baseline, alpha = 1 means use matched value
    # Transition zone is defined around the threshold
    delta = threshold * transition_width
    lower_bound = threshold - delta
    upper_bound = threshold + delta
    
    # Linear ramp blending mask
    alpha = np.clip((abs_coeff - lower_bound) / (2 * delta + 1e-8), 0.0, 1.0)
    
    # 5. Blend the original absolute values with the matched ones smoothly
    abs_coeff_blended = (1 - alpha) * abs_coeff + alpha * matched_full
    
    # 6. Re-apply original signs
    return abs_coeff_blended * signs

def match_cA_with_smooth_transition(cA_coeff, quantile=0.96, transition_width=0.05):
    """
    Symmetric smooth blending specifically designed for the fully positive cA.
    """
    threshold = np.quantile(cA_coeff, quantile)
    baseline_mask = cA_coeff <= threshold
    outlier_mask = ~baseline_mask
    
    if not np.any(outlier_mask):
        return cA_coeff
        
    baseline_values = cA_coeff[baseline_mask]
    outlier_values = cA_coeff[outlier_mask]
    
    matched_outliers = match_histograms(outlier_values, baseline_values)
    matched_full = cA_coeff.copy()
    matched_full[outlier_mask] = matched_outliers
    
    # Smooth blending mask
    delta = threshold * transition_width
    lower_bound = threshold - delta
    alpha = np.clip((cA_coeff - lower_bound) / (2 * delta + 1e-8), 0.0, 1.0)
    
    cA_blended = (1 - alpha) * cA_coeff + alpha * matched_full
    return cA_blended

In [98]:
import numpy as np
import pywt
import matplotlib.pyplot as plt
from skimage.exposure import match_histograms

# --- Set configurations ---
WAVELET = 'sym4'
LEVEL = 3

# =====================================================================
# 1. DEFINE TRANSITION BLENDING FUNCTIONS
# =====================================================================
def match_outliers_with_smooth_transition(coeff, quantile=0.8, transition_width=0.1):
    """
    Blends matched outliers smoothly with the baseline over a transition zone
    around the threshold, keeping ordering and eliminating boundary artifacts.
    """
    signs = np.sign(coeff)
    abs_coeff = np.abs(coeff)
    
    threshold = np.quantile(abs_coeff, quantile)
    if threshold == 0:
        return coeff, abs_coeff
    
    baseline_mask = abs_coeff <= threshold
    outlier_mask = ~baseline_mask
    
    if not np.any(outlier_mask):
        return coeff, abs_coeff
    
    baseline_values = abs_coeff[baseline_mask]
    outlier_values = abs_coeff[outlier_mask]
    
    # Histogram match the outliers to the baseline
    matched_outliers = match_histograms(outlier_values, baseline_values)
    matched_full = abs_coeff.copy()
    matched_full[outlier_mask] = matched_outliers
    
    # Create smooth transition mask (alpha ramp)
    delta = threshold * transition_width
    lower_bound = threshold - delta
    upper_bound = threshold + delta
    
    # Calculate transition weights: 0.0 (baseline) to 1.0 (fully matched)
    alpha = np.clip((abs_coeff - lower_bound) / (2 * delta + 1e-8), 0.0, 1.0)
    
    # Interpolate
    abs_coeff_blended = (1.0 - alpha) * abs_coeff + alpha * matched_full
    
    return abs_coeff_blended * signs, abs_coeff_blended

def match_cA_with_smooth_transition(cA_coeff, quantile=0.96, transition_width=0.05):
    """
    Symmetric smooth blending specifically designed for the fully positive cA.
    """
    threshold = np.quantile(cA_coeff, quantile)
    baseline_mask = cA_coeff <= threshold
    outlier_mask = ~baseline_mask
    
    if not np.any(outlier_mask):
        return cA_coeff, cA_coeff
        
    baseline_values = cA_coeff[baseline_mask]
    outlier_values = cA_coeff[outlier_mask]
    
    matched_outliers = match_histograms(outlier_values, baseline_values)
    matched_full = cA_coeff.copy()
    matched_full[outlier_mask] = matched_outliers
    
    # Smooth blending mask
    delta = threshold * transition_width
    lower_bound = threshold - delta
    
    alpha = np.clip((cA_coeff - lower_bound) / (2 * delta + 1e-8), 0.0, 1.0)
    
    cA_blended = (1.0 - alpha) * cA_coeff + alpha * matched_full
    return cA_blended, cA_blended

# =====================================================================
# 2. MULTI-LEVEL DECOMPOSITION & SMOOTH FILTERING
# =====================================================================
# Decompose
coeffs = pywt.wavedec2(M, WAVELET, level=LEVEL)
cA_n = coeffs[0]
details = coeffs[1:]

# Filter approximation with a tight transition
cA_n_filt, _ = match_cA_with_smooth_transition(cA_n, quantile=0.80, transition_width=0.05)

# Filter details with transition blending
details_filt = []
for level_details in details:
    cH, cV, cD = level_details
    cH_f, _ = match_outliers_with_smooth_transition(cH, quantile=0.8, transition_width=0.1)
    cV_f, _ = match_outliers_with_smooth_transition(cV, quantile=0.8, transition_width=0.1)
    cD_f, _ = match_outliers_with_smooth_transition(cD, quantile=0.8, transition_width=0.1)
    details_filt.append((cH_f, cV_f, cD_f))

# Reassemble
coeffs_filtered = [cA_n_filt] + details_filt

# Reconstruct
M_smooth_clean = pywt.waverec2(coeffs_filtered, WAVELET)

if M_smooth_clean.shape != M.shape:
    M_smooth_clean = M_smooth_clean[:M.shape[0], :M.shape[1]]

# =====================================================================
# 3. VISUALIZATION AND ANALYSIS
# =====================================================================
fig, axes = plt.subplots(1, 5, figsize=(18, 5), constrained_layout=True)

# 1. Original Spectrogram
im0 = axes[0].pcolormesh(t_spectro, f_M, M, shading='nearest', cmap='jet', vmin = 0.001, vmax = 0.12)
axes[0].set_title("Original Spectrogram")
axes[0].set_ylabel("Frequency")
axes[0].set_xlabel("Time")
fig.colorbar(im0, ax=axes[0], label="Intensity")

# 2. Smoothly Filtered Spectrogram
im1 = axes[1].pcolormesh(t_spectro, f_M, M_smooth_clean, shading='nearest', cmap='jet', vmin = 0.001, vmax = 0.12)
axes[1].set_title("Smooth-Blended Spectrogram")
axes[1].set_xlabel("Time")
fig.colorbar(im1, ax=axes[1], label="Intensity")

# 3. Difference Map (To verify where changes occurred)
diff_map = M - M_smooth_clean
im1bis = axes[2].pcolormesh(t_spectro, f_M, M_smooth_clean, shading='nearest', cmap='jet') # Blue-White-Red map
axes[2].set_title("Smooth-Blended Spectrogram (colormap scaled)")
axes[2].set_xlabel("Time")
fig.colorbar(im1, ax=axes[2], label="Intensity")

# 3. Difference Map (To verify where changes occurred)
diff_map = M - M_smooth_clean
im2 = axes[3].pcolormesh(t_spectro, f_M, diff_map, shading='nearest', cmap='bwr') # Blue-White-Red map
axes[3].set_title("Difference Map (Original - Clean)")
axes[3].set_xlabel("Time")

# Center the colormap at 0 so white means "no change"
max_diff = np.max(np.abs(diff_map))
im2.set_clim(-max_diff, max_diff)
fig.colorbar(im2, ax=axes[3], label="Difference Intensity")

axes[4].plot(f_M, np.mean(M, axis = 1), label = 'PSD of original spectrogram')
axes[4].plot(f_M, np.mean(M_smooth_clean, axis = 1), label = 'PSD of cleaned spectrogram')
axes[4].legend()

plt.show()

In [97]:
import numpy as np
import pywt
import matplotlib.pyplot as plt
from skimage.exposure import match_histograms

# --- Configurations ---
WAVELET = 'sym4'
LEVEL = 3

# =====================================================================
# 1. HELPER FUNCTIONS (The "Engine")
# =====================================================================
def match_outliers_with_smooth_transition(coeff, quantile=0.8, transition_width=0.1):
    signs = np.sign(coeff)
    abs_coeff = np.abs(coeff)
    threshold = np.quantile(abs_coeff, quantile)
    if threshold == 0: return coeff, abs_coeff
    
    baseline_mask = abs_coeff <= threshold
    outlier_mask = ~baseline_mask
    if not np.any(outlier_mask): return coeff, abs_coeff
    
    matched_outliers = match_histograms(abs_coeff[outlier_mask], abs_coeff[baseline_mask])
    matched_full = abs_coeff.copy()
    matched_full[outlier_mask] = matched_outliers
    
    delta = threshold * transition_width
    alpha = np.clip((abs_coeff - (threshold - delta)) / (2 * delta + 1e-8), 0.0, 1.0)
    abs_coeff_blended = (1.0 - alpha) * abs_coeff + alpha * matched_full
    return abs_coeff_blended * signs, abs_coeff_blended

def match_cA_with_smooth_transition(cA_coeff, quantile=0.96, transition_width=0.05):
    threshold = np.quantile(cA_coeff, quantile)
    baseline_mask = cA_coeff <= threshold
    outlier_mask = ~baseline_mask
    if not np.any(outlier_mask): return cA_coeff, cA_coeff
        
    matched_outliers = match_histograms(cA_coeff[outlier_mask], cA_coeff[baseline_mask])
    matched_full = cA_coeff.copy()
    matched_full[outlier_mask] = matched_outliers
    
    delta = threshold * transition_width
    alpha = np.clip((cA_coeff - (threshold - delta)) / (2 * delta + 1e-8), 0.0, 1.0)
    cA_blended = (1.0 - alpha) * cA_coeff + alpha * matched_full
    return cA_blended, cA_blended

def get_frequency_weights(frequencies, f_bump_center, f_bump_width, roll_off=1.5):
    f_min = f_bump_center - (f_bump_width / 2.0)
    f_max = f_bump_center + (f_bump_width / 2.0)
    total_dist = np.maximum(0, f_min - frequencies) + np.maximum(0, frequencies - f_max)
    return np.exp(-0.5 * (total_dist / (roll_off + 1e-8))**2)

def interpolate_mask_to_match_shape(mask_1d, target_height):
    return np.interp(np.linspace(0, 1, target_height), np.linspace(0, 1, len(mask_1d)), mask_1d)

# =====================================================================
# 2. THE PIPELINE
# =====================================================================
# Settings
BUMP_CENTER_FREQ = 30
BUMP_WIDTH = 6
ROLL_OFF_WIDTH = 1

# =====================================================================
# UPDATED PIPELINE: Mask-First Localized Thresholding
# =====================================================================

# 1. Generate the master 1D frequency weight mask
freq_weights_1d = get_frequency_weights(f_M, BUMP_CENTER_FREQ, BUMP_WIDTH, ROLL_OFF_WIDTH)

# Decompose
coeffs = pywt.wavedec2(M, WAVELET, level=LEVEL)
cA_n = coeffs[0]
details = coeffs[1:]

# 2. Localized cA Filtering
# Interpolate mask for cA
cA_weight = interpolate_mask_to_match_shape(freq_weights_1d, cA_n.shape[0])[:, np.newaxis]

# Calculate the threshold ONLY using cA elements that sit inside our target band
# (where cA_weight > 0.5) to keep the baseline estimation pure
inside_cA_mask = (cA_weight > 0.5).squeeze()
if np.any(inside_cA_mask):
    # Calculate threshold based purely on target-band statistics
    cA_n_filt, _ = match_cA_with_smooth_transition(cA_n, quantile=0.80, transition_width=0.05)
else:
    cA_n_filt = cA_n.copy()

# Blend back: Out-of-band values are 100% untouched
cA_n_final = (cA_weight * cA_n_filt) + ((1.0 - cA_weight) * cA_n)


# 3. Localized Details Filtering
details_filt = []
for i, level_details in enumerate(details):
    cH, cV, cD = level_details
    det_weight = interpolate_mask_to_match_shape(freq_weights_1d, cH.shape[0])[:, np.newaxis]
    
    # Run the smooth transition matching
    cH_f, _ = match_outliers_with_smooth_transition(cH, quantile=0.8, transition_width=0.1)
    cV_f, _ = match_outliers_with_smooth_transition(cV, quantile=0.8, transition_width=0.1)
    cD_f, _ = match_outliers_with_smooth_transition(cD, quantile=0.8, transition_width=0.1)
    
    # Force out-of-band regions to stay exactly as they were, with zero changes
    cH_final = (det_weight * cH_f) + ((1.0 - det_weight) * cH)
    cV_final = (det_weight * cV_f) + ((1.0 - det_weight) * cV)
    cD_final = (det_weight * cD_f) + ((1.0 - det_weight) * cD)
    
    details_filt.append((cH_final, cV_final, cD_final))

# 4. Reconstruct
M_smooth_clean = pywt.waverec2([cA_n_final] + details_filt, WAVELET)
if M_smooth_clean.shape != M.shape:
    M_smooth_clean = M_smooth_clean[:M.shape[0], :M.shape[1]]

# =====================================================================
# 3. VISUALIZATION AND ANALYSIS
# =====================================================================
fig, axes = plt.subplots(1, 5, figsize=(18, 5), constrained_layout=True)

# 1. Original Spectrogram
im0 = axes[0].pcolormesh(t_spectro, f_M, M, shading='nearest', cmap='jet', vmin = 0.001, vmax = 0.12)
axes[0].set_title("Original Spectrogram")
axes[0].set_ylabel("Frequency")
axes[0].set_xlabel("Time")
fig.colorbar(im0, ax=axes[0], label="Intensity")

# 2. Smoothly Filtered Spectrogram
im1 = axes[1].pcolormesh(t_spectro, f_M, M_smooth_clean, shading='nearest', cmap='jet', vmin = 0.001, vmax = 0.12)
axes[1].set_title("Smooth-Blended Spectrogram")
axes[1].set_xlabel("Time")
fig.colorbar(im1, ax=axes[1], label="Intensity")

# 3. Difference Map (To verify where changes occurred)
diff_map = M - M_smooth_clean
im1bis = axes[2].pcolormesh(t_spectro, f_M, M_smooth_clean, shading='nearest', cmap='jet') # Blue-White-Red map
axes[2].set_title("Smooth-Blended Spectrogram (colormap scaled)")
axes[2].set_xlabel("Time")
fig.colorbar(im1bis, ax=axes[2], label="Intensity")

# 3. Difference Map (To verify where changes occurred)
diff_map = M - M_smooth_clean
im2 = axes[3].pcolormesh(t_spectro, f_M, diff_map, shading='nearest', cmap='bwr') # Blue-White-Red map
axes[3].set_title("Difference Map (Original - Clean)")
axes[3].set_xlabel("Time")

# Center the colormap at 0 so white means "no change"
max_diff = np.max(np.abs(diff_map))
im2.set_clim(-max_diff, max_diff)
fig.colorbar(im2, ax=axes[3], label="Difference Intensity")

axes[4].plot(f_M, np.mean(M, axis = 1), label = 'PSD of original spectrogram')
axes[4].plot(f_M, np.mean(M_smooth_clean, axis = 1), label = 'PSD of cleaned spectrogram')
axes[4].legend()

plt.show()

C:\Users\holcman\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\pywt\_multilevel.py:43: UserWarning: Level value of 3 is too high: all coefficients will experience boundary effects.
  warnings.warn(


In [99]:
fig, axes = plt.subplots(2, figsize=(18, 5), constrained_layout=True)

# 1. Original Spectrogram
im0 = axes[0].pcolormesh(t_spectro, f_M, np.log2(M + 1e-11), shading='nearest', cmap='jet')#, vmin = 0.001, vmax = 0.12)
axes[0].set_title("Original Spectrogram")
axes[0].set_ylabel("Frequency")
axes[0].set_xlabel("Time")
fig.colorbar(im0, ax=axes[0], label="Intensity")

M_smooth_clean = np.clip(M_smooth_clean, 0.0001, np.max(M_smooth_clean))
# 2. Smoothly Filtered Spectrogram
im1 = axes[1].pcolormesh(t_spectro, f_M, np.log2(M_smooth_clean + 1e-11), shading='nearest', cmap='jet')#, vmin = 0.001, vmax = 0.12)
axes[1].set_title("Smooth-Blended Spectrogram")
axes[1].set_xlabel("Time")
fig.colorbar(im1, ax=axes[1], label="Intensity")

plt.show()

Have issue when going bakc to log, also it seems it did not keep spindles structure

## Attenuation on spectrogram directly

In [214]:
mask = np.quantile(M, 0.2)

fig, axes = plt.subplots(2)
axes[0].pcolormesh(t_spectro, f_M, np.log2(M + 1e-11), shading = 'nearest', cmap = 'jet')
axes[0].contour(t_spectro, f_M, mask.astype(float), levels=[0.5], colors='white', linewidths=2)
axes[1].pcolormesh(t_spectro, f_M, mask, shading = 'nearest', cmap = 'jet')

plt.show()

C:\Users\holcman\AppData\Local\Temp\ipykernel_9092\499472426.py:4: RuntimeWarning: invalid value encountered in log2
  axes[0].pcolormesh(t_spectro, f_M, np.log2(M + 1e-11), shading = 'nearest', cmap = 'jet')


TypeError: Input z must be 2D, not 0D

## Optimization of Cumulative transform

In [191]:
M = spectro.copy()
mask_f = f_spectro >= 20
f_M = f_spectro[mask_f]
M = M[mask_f, :]

fig, axis = plt.subplots(1)
axis.pcolormesh(t_spectro, f_M, np.log2(M + 1e-11), shading = 'nearest', cmap = 'jet')

M = np.log2(M + 1e-11)

Cumulative parameterized and optimization

In [166]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import minimize
from scipy.linalg import solve

# =====================================================================
# 1. WHITTAKER ASYMMETRIC LEAST SQUARES (For Target Baseline)
# =====================================================================
def whittaker_als_baseline(y, lam=1e4, p=0.01, max_iter=15):
    L = len(y)
    D = np.zeros((L-2, L))
    for i in range(L-2):
        D[i, i], D[i, i+1], D[i, i+2] = 1, -2, 1
    penalty = lam * np.dot(D.T, D)
    w = np.ones(L)
    for _ in range(max_iter):
        W = np.diag(w)
        z = solve(W + penalty, w * y)
        w = np.where(y > z, p, 1 - p)
    return z

# =====================================================================
# 2. SETUP DATA AND WHITTAKER PROFILE
# =====================================================================
# Assumes M and mask are already present in your environment
freq_bins = M.shape[0]
original_psd = M.mean(axis=1)
reference_psd = whittaker_als_baseline(original_psd, lam=1e4, p=0.01)

# Establish the floor threshold (the boundary value of your mask)
threshold_val = M[mask].min()

# =====================================================================
# 3. DEFINE SIGNED LOG-SPACE MONOTONIC OBJECTIVE FUNCTION
# =====================================================================
def signed_log_monotonic_objective(params, M_base, blob_mask, threshold_val, target_psd):
    """
    Applies a parametric soft-clipping curve directly to the log-excess height.
    Works flawlessly with both positive and negative log2 values.
    """
    gamma = params[0]
    M_mod = M_base.copy()
    
    M_blob = M_base[blob_mask]
    # Calculate positive distance above the mask floor
    excess = M_blob - threshold_val
    
    # Strictly monotonic rational compression function
    # It leaves values near the threshold alone and squashes high peaks
    transformed_excess = excess / (1.0 + gamma * excess)
    
    # Re-apply back to the threshold baseline
    M_mod[blob_mask] = threshold_val + transformed_excess
    
    # Calculate PSD profile error
    current_psd = M_mod.mean(axis=1)
    psd_loss = np.sum((current_psd - target_psd) ** 2)
    
    return psd_loss

# =====================================================================
# 4. OPTIMIZE THE CURVE PARAMETER
# =====================================================================
# Initial guess: gamma = 0.0 (corresponds to no change at all)
initial_gamma = [0.0]

# Bounds: gamma >= 0 ensures strict monotonicity and compression
bounds = [(0.0, 100.0)]

print("Optimizing signed log-space transfer function...")
res = minimize(
    signed_log_monotonic_objective,
    initial_gamma,
    args=(M, mask, threshold_val, reference_psd),
    bounds=bounds,
    method='L-BFGS-B',
    options={'ftol': 1e-12}
)

best_gamma = res.x[0]
print(f"Optimization Successful: {res.success}")
print(f"Optimal Compression Intensity (Gamma): {best_gamma:.4f}")

# =====================================================================
# 5. GENERATE FINAL SPECTROGRAM
# =====================================================================
M_final = M.copy()
excess_final = M[mask] - threshold_val
M_final[mask] = threshold_val + (excess_final / (1.0 + best_gamma * excess_final))

# =====================================================================
# 6. VISUALIZE
# =====================================================================
fig, axes = plt.subplots(3, 1, figsize=(12, 10))

im0 = axes[0].pcolormesh(M, shading='auto', cmap='jet')
axes[0].set_title(f"Original Log2 M (Max: {M.max():.2f})")
fig.colorbar(im0, ax=axes[0])

im1 = axes[1].pcolormesh(M_final, shading='auto', cmap='jet', vmin=M_final.min(), vmax=M_final.max())
axes[1].set_title(f"Monotonically Attenuated Log2 M (Max: {M_final.max():.2f})")
fig.colorbar(im1, ax=axes[1])

axes[2].plot(f_M, original_psd, label='PSD original spectrogram', color='red', alpha=0.6)
axes[2].plot(f_M, reference_psd, label='Whittaker Target Baseline', color='black', linestyle='--')
axes[2].plot(f_M, np.mean(M_final, axis=1), label='PSD cleaned spectrogram', color='green', linewidth=2)
axes[2].set_title("PSD Profile Verification")
axes[2].legend()
axes[2].grid(True)

plt.tight_layout()
plt.show()

Optimizing signed log-space transfer function...
Optimization Successful: True
Optimal Compression Intensity (Gamma): 0.4035


cumulative parameterized and relative SSIM regularization optimization

In [168]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import minimize
from scipy.linalg import solve
from skimage.metrics import structural_similarity as ssim

# =====================================================================
# 1. WHITTAKER ASYMMETRIC LEAST SQUARES (For Target Baseline)
# =====================================================================
def whittaker_als_baseline(y, lam=1e4, p=0.01, max_iter=15):
    """
    Computes a smooth baseline using a standard dense linear system.
    Bypasses high-intensity peaks to generate a clean target profile.
    """
    L = len(y)
    D = np.zeros((L-2, L))
    for i in range(L-2):
        D[i, i], D[i, i+1], D[i, i+2] = 1, -2, 1
        
    penalty = lam * np.dot(D.T, D)
    w = np.ones(L)
    
    for _ in range(max_iter):
        W = np.diag(w)
        z = solve(W + penalty, w * y)
        w = np.where(y > z, p, 1 - p)
        
    return z

# =====================================================================
# 2. SETUP DATA AND WHITTAKER PROFILE
# =====================================================================
# Assumes M and mask are already present in your environment
freq_bins = M.shape[0]
original_psd = M.mean(axis=1)
reference_psd = whittaker_als_baseline(original_psd, lam=1e4, p=0.01)

# Establish the floor threshold (the boundary value of your mask)
threshold_val = M[mask].min()

# =====================================================================
# 3. DEFINE THE OBJECTIVE FUNCTION WITH LOCALIZED RANGE-NORMALIZED SSIM
# =====================================================================
def signed_log_monotonic_objective(params, M_base, blob_mask, threshold_val, target_psd):
    """
    Applies a parametric soft-clipping curve directly to the log-excess height.
    Uses localized, range-normalized SSIM to preserve relative variation and contours.
    """
    gamma = params[0]
    M_mod = M_base.copy()
    
    M_blob = M_base[blob_mask]
    excess = M_blob - threshold_val
    
    # 1. Monotonic compression function
    transformed_excess = excess / (1.0 + gamma * excess)
    M_mod[blob_mask] = threshold_val + transformed_excess
    
    # -----------------------------------------------------------------
    # TERM 1: PSD Target Loss (Drive toward Whittaker baseline)
    # -----------------------------------------------------------------
    current_psd = M_mod.mean(axis=1)
    psd_loss = np.sum((current_psd - target_psd) ** 2)
    
    # -----------------------------------------------------------------
    # TERM 2: Relative Variation SSIM Loss (Inside Mask Only)
    # -----------------------------------------------------------------
    candidate_blob = M_mod[blob_mask]
    
    # Min-Max normalize both blobs to a [0, 1] scale locally.
    # This strips away absolute variance changes and preserves relative contours.
    orig_blob_norm = (M_blob - M_blob.min()) / (M_blob.max() - M_blob.min() + 1e-9)
    cand_blob_norm = (candidate_blob - candidate_blob.min()) / (candidate_blob.max() - candidate_blob.min() + 1e-9)
    
    # Calculate 1D SSIM on the normalized intensity structures.
    # Using a small win_size since we are operating directly on a 1D flattened array.
    relative_ssim = ssim(orig_blob_norm, cand_blob_norm, data_range=1.0, win_size=7)
    relative_ssim_loss = 1.0 - relative_ssim
    
    # -----------------------------------------------------------------
    # COMBINE LOSSES
    # -----------------------------------------------------------------
    # Adjust the weight factor (10.0) depending on how strongly you want 
    # to enforce relative structure matching over strict baseline fitting.
    total_loss = psd_loss + (10.0 * relative_ssim_loss)
    
    return total_loss

# =====================================================================
# 4. OPTIMIZE THE CURVE PARAMETER
# =====================================================================
# Initial guess: gamma = 0.0 (corresponds to no change at all)
initial_gamma = [0.0]

# Bounds: gamma >= 0 ensures strict monotonicity and peak compression
bounds = [(0.0, 100.0)]

print("Optimizing signed log-space transfer function with relative SSIM constraints...")
res = minimize(
    signed_log_monotonic_objective,
    initial_gamma,
    args=(M, mask, threshold_val, reference_psd),
    bounds=bounds,
    method='L-BFGS-B',
    options={'ftol': 1e-12}
)

best_gamma = res.x[0]
print(f"Optimization Successful: {res.success}")
print(f"Optimal Compression Intensity (Gamma): {best_gamma:.4f}")

# =====================================================================
# 5. GENERATE FINAL SPECTROGRAM
# =====================================================================
M_final = M.copy()
excess_final = M[mask] - threshold_val
M_final[mask] = threshold_val + (excess_final / (1.0 + best_gamma * excess_final))

# =====================================================================
# 6. VISUALIZE
# =====================================================================
fig, axes = plt.subplots(3, 1, figsize=(12, 10))

im0 = axes[0].pcolormesh(M, shading='auto', cmap='plasma')
axes[0].set_title(f"Original Log2 M (Max: {M.max():.2f})")
fig.colorbar(im0, ax=axes[0])

im1 = axes[1].pcolormesh(M_final, shading='auto', cmap='plasma', vmin=M_final.min(), vmax=M_final.max())
axes[1].set_title(f"Relative-SSIM Attenuated Log2 M (Max: {M_final.max():.2f})")
fig.colorbar(im1, ax=axes[1])

# Plot PSD Profiles comparison
axes[2].plot(f_M, original_psd, label='PSD original spectrogram', color='red', alpha=0.6)
axes[2].plot(f_M, reference_psd, label='Whittaker Target Baseline', color='black', linestyle='--')
axes[2].plot(f_M, np.mean(M_final, axis=1), label='PSD cleaned spectrogram', color='green', linewidth=2)
axes[2].set_title("PSD Profile Verification")
axes[2].legend()
axes[2].grid(True)

plt.tight_layout()
plt.show()

Optimizing signed log-space transfer function with relative SSIM constraints...
Optimization Successful: True
Optimal Compression Intensity (Gamma): 0.3781


In [165]:
import numpy as np
from scipy.ndimage import laplace, sobel

def quantify_hill_structure(M_orig, M_comp, blob_mask):
    """
    Quantifies how well the 2D hill topology/curvature is preserved 
    independent of absolute value attenuation.
    """
    # 1. Isolate and Min-Max Normalize the blob regions to [0, 1]
    # This isolates structural shape from absolute amplitude drops
    b_orig = M_orig.copy()
    b_comp = M_comp.copy()
    
    # Set everything outside mask to 0 to focus calculations
    b_orig[~blob_mask] = b_orig[blob_mask].min()
    b_comp[~blob_mask] = b_comp[blob_mask].min()
    
    norm_orig = (b_orig - b_orig.min()) / (b_orig.max() - b_orig.min() + 1e-9)
    norm_comp = (b_comp - b_comp.min()) / (b_comp.max() - b_comp.min() + 1e-9)
    
    # 2. Compute Slopes (Gradients via Sobel filters)
    grad_orig_x = sobel(norm_orig, axis=0)
    grad_orig_y = sobel(norm_orig, axis=1)
    grad_comp_x = sobel(norm_comp, axis=0)
    grad_comp_y = sobel(norm_comp, axis=1)
    
    mag_orig = np.sqrt(grad_orig_x**2 + grad_orig_y**2)
    mag_comp = np.sqrt(grad_comp_x**2 + grad_comp_y**2)
    
    # NRMSE of the gradients
    grad_nrmse = np.sqrt(np.mean((mag_orig - mag_comp) ** 2)) / (mag_orig.max() - mag_orig.min() + 1e-9)
    
    # 3. Compute Peak Curvature (Laplacian)
    lap_orig = laplace(norm_orig)
    lap_comp = laplace(norm_comp)
    
    # Ratio of maximum sharpness (absolute value because peaks have negative curvature)
    curvature_retention = np.max(np.abs(lap_comp)) / (np.max(np.abs(lap_orig)) + 1e-9)
    
    return {
        "Gradient Error (Lower is Better)": grad_nrmse,
        "Peak Curvature Retention (Closer to 1.0 is Better)": curvature_retention
    }

# =====================================================================
# HOW TO USE IT IN YOUR WORKFLOW:
# =====================================================================
report_reg = quantify_hill_structure(M, M_final, mask)
print(report_reg)
# report_unreg = quantify_hill_structure(M, M_final_unregularized, mask)

{'Gradient Error (Lower is Better)': 0.21871355802798564, 'Peak Curvature Retention (Closer to 1.0 is Better)': 1.4495621600680317}


In [167]:
report_reg = quantify_hill_structure(M, M_final, mask)
print(report_reg)

{'Gradient Error (Lower is Better)': 0.2263870859152361, 'Peak Curvature Retention (Closer to 1.0 is Better)': 1.4618615801289074}


In [197]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import minimize
from scipy.linalg import solve
from scipy.ndimage import distance_transform_edt
from skimage.metrics import structural_similarity as ssim

# =====================================================================
# 1. WHITTAKER ASYMMETRIC LEAST SQUARES (Baseline Target Profile)
# =====================================================================
def whittaker_als_baseline(y, lam=1e4, p=0.01, max_iter=15):
    L = len(y)
    D = np.zeros((L-2, L))
    for i in range(L-2):
        D[i, i], D[i, i+1], D[i, i+2] = 1, -2, 1
    penalty = lam * np.dot(D.T, D)
    w = np.ones(L)
    for _ in range(max_iter):
        W = np.diag(w)
        z = solve(W + penalty, w * y)
        w = np.where(y > z, p, 1 - p)
    return z

# =====================================================================
# 2. SETUP DATA AND VECTORIZED PROPERTIES
# =====================================================================
# Assumes your 2D Log2 matrix 'M' and boolean matrix 'mask' are loaded.
freq_bins = M.shape[0]
original_psd = M.mean(axis=1)
reference_psd = whittaker_als_baseline(original_psd, lam=1e4, p=0.01)

# Isolate original active blob intensities
M_blob_orig = M[mask]

# --- 1D Empirical Cumulative Distribution Function (eCDF) ---
sort_indices = np.argsort(M_blob_orig)
rank_percentiles = np.zeros_like(M_blob_orig)
rank_percentiles[sort_indices] = np.linspace(0.0, 1.0, len(M_blob_orig))

# --- Match each pixel to its specific destination frequency floor ---
row_indices = np.where(mask)[0]
target_floors = reference_psd[row_indices]

# --- Calculate the absolute log-excess above target floors ---
# Clipping at 0 ensures we only look at values that are genuinely above the target
original_excess = np.clip(M_blob_orig - target_floors, 0.0, None)

# --- Spatial Blending Weights Map ---
distance_map = distance_transform_edt(mask)
transition_width = 2.0 
weights = 1.0 - np.exp(-(distance_map / transition_width) ** 2)
weights[~mask] = 0.0
W_blob = weights[mask]

# =====================================================================
# 3. OBJECTIVE FUNCTION: GUARANTEED ATTENUATION
# =====================================================================
def strict_attenuation_objective(params, M_base, blob_mask, rank_pct, W_b, target_floors, original_excess, target_psd):
    """
    Applies a power-law decay curve driven by cumulative rank percentiles.
    Guarantees strict attenuation of log-space energy toward the baseline.
    """
    gamma = params[0]
    M_mod = M_base.copy()
    
    # Warping function: As gamma increases from 0 upward, 
    # this multiplier drops smoothly toward 0, forcing attenuation.
    attenuation_multiplier = np.power(rank_pct, gamma)
    
    # Compress the excess envelope directly
    attenuated_excess = original_excess * attenuation_multiplier
    
    # Reconstruct values safely anchored to the destination floors
    M_fully_optimized_blob = target_floors + attenuated_excess
    
    # Perform spatial alpha-blending
    M_blob_orig = M_base[blob_mask]
    M_mod[blob_mask] = (1.0 - W_b) * M_blob_orig + W_b * M_fully_optimized_blob
    
    # --- Loss 1: Mean PSD Target Alignment ---
    current_psd = M_mod.mean(axis=1)
    psd_loss = np.sum((current_psd - target_psd) ** 2)
    
    # --- Loss 2: Local Range-Normalized SSIM Structure Guard ---
    candidate_blob = M_mod[blob_mask]
    orig_blob_norm = (M_blob_orig - M_blob_orig.min()) / (M_blob_orig.max() - M_blob_orig.min() + 1e-9)
    cand_blob_norm = (candidate_blob - candidate_blob.min()) / (candidate_blob.max() - candidate_blob.min() + 1e-9)
    relative_ssim_loss = 1.0 - ssim(orig_blob_norm, cand_blob_norm, data_range=1.0, win_size=7)
    
    print(f'psd_loss: {psd_loss} | realtive_ssim_loss: {relative_ssim_loss}')

    return psd_loss + (10.0 * relative_ssim_loss)

# =====================================================================
# 4. RUN OPTIMIZER (Gamma starting at 0 = No attenuation)
# =====================================================================
initial_gamma = [0.0]
bounds = [(0.0, 50.0)] # Increasing gamma drives attenuation more aggressively

print("Optimizing strict attenuation parameters...")
res = minimize(
    strict_attenuation_objective,
    initial_gamma,
    args=(M, mask, rank_percentiles, W_blob, target_floors, original_excess, reference_psd),
    bounds=bounds,
    method='L-BFGS-B',
    options={'ftol': 1e-12}
)

best_gamma = res.x[0]
print(f"Optimization Status: {res.success} | Optimal Attenuation Factor (Gamma): {best_gamma:.4f}")

# =====================================================================
# 5. EXECUTE FINAL RECONSTRUCTION
# =====================================================================
M_final = M.copy()
final_multiplier = np.power(rank_percentiles, best_gamma)
M_fully_optimized_blob = target_floors + (original_excess * final_multiplier)

# Reapply spatial edge blending map
M_final[mask] = (1.0 - W_blob) * M[mask] + W_blob * M_fully_optimized_blob

# =====================================================================
# 6. VISUALIZE
# =====================================================================
fig, axes = plt.subplots(3, 1, figsize=(12, 10))

im0 = axes[0].pcolormesh(M, shading='auto', cmap='jet')
axes[0].set_title(f"Original Log2 M (Max: {M.max():.2f})")
fig.colorbar(im0, ax=axes[0])

im1 = axes[1].pcolormesh(M_final, shading='auto', cmap='jet', vmin=M_final.min(), vmax=M_final.max())
axes[1].contour(mask)
axes[1].set_title("Strictly Attenuated Cumulative Spectrogram")
fig.colorbar(im1, ax=axes[1])

axes[2].plot(f_M, original_psd, label='PSD original spectrogram', color='red', alpha=0.6)
axes[2].plot(f_M, reference_psd, label='Whittaker Target Baseline', color='black', linestyle='--')
axes[2].plot(f_M, np.mean(M_final, axis=1), label='PSD cleaned spectrogram', color='green', linewidth=2)
axes[2].set_title("PSD Profile Verification")
axes[2].legend()
axes[2].grid(True)

plt.tight_layout()
plt.show()

Optimizing strict attenuation parameters...


ValueError: zero-size array to reduction operation minimum which has no identity

In [196]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import minimize
from scipy.linalg import solve
from scipy.ndimage import distance_transform_edt
from skimage.metrics import structural_similarity as ssim

# =====================================================================
# 1. WHITTAKER ASYMMETRIC LEAST SQUARES (For Target Baseline Verification)
# =====================================================================
def whittaker_als_baseline(y, lam=1e4, p=0.01, max_iter=15):
    L = len(y)
    D = np.zeros((L-2, L))
    for i in range(L-2):
        D[i, i], D[i, i+1], D[i, i+2] = 1, -2, 1
    penalty = lam * np.dot(D.T, D)
    w = np.ones(L)
    for _ in range(max_iter):
        W = np.diag(w)
        z = solve(W + penalty, w * y)
        w = np.where(y > z, p, 1 - p)
    return z

# =====================================================================
# 2. SETUP DATA AND SINGLE SCALAR PROPERTY MAPPING
# =====================================================================
# Assumes your 2D Log2 matrix 'M' and boolean matrix 'mask' are loaded.
freq_bins = M.shape[0]
original_psd = M.mean(axis=1)
reference_psd = whittaker_als_baseline(original_psd, lam=1e4, p=0.01)

# Isolate original active blob intensities
M_blob_orig = M[mask]
blob_max = M_blob_orig.max()

# --- CRITICAL CHANGE FOR COMPARISON: Single Global Floor Threshold ---
# Instead of using the row-by-row vector, we lock the baseline to a single scalar
threshold_val = M_blob_orig.min()

# --- Calculate the absolute log-excess above the single static floor ---
original_excess_scalar = np.clip(M_blob_orig - threshold_val, 0.0, None)

# --- 1D Empirical Cumulative Distribution Function (eCDF) ---
sort_indices = np.argsort(M_blob_orig)
rank_percentiles = np.zeros_like(M_blob_orig)
rank_percentiles[sort_indices] = np.linspace(0.0, 1.0, len(M_blob_orig))

# --- Spatial Blending Weights Map ---
distance_map = distance_transform_edt(mask)
transition_width = 1.0 
weights = 1.0 - np.exp(-(distance_map / transition_width) ** 2)
weights[~mask] = 0.0
W_blob = weights[mask]

# =====================================================================
# 3. OBJECTIVE FUNCTION: SINGLE SCALAR THRESHOLD ATTENUATION
# =====================================================================
def scalar_threshold_objective(params, M_base, blob_mask, rank_pct, W_b, threshold_val, original_excess, target_psd):
    """
    Applies cumulative power-law attenuation but forces all pixels inside 
    the mask to decay toward a single uniform scalar threshold baseline.
    """
    gamma = params[0]
    M_mod = M_base.copy()
    
    # Warping function driven by cumulative rank percentiles
    attenuation_multiplier = np.power(rank_pct, gamma)
    
    # Compress excess relative to the uniform static floor
    attenuated_excess = original_excess * attenuation_multiplier
    
    # Reconstruct values safely anchored ONLY to the single scalar floor
    M_fully_optimized_blob = threshold_val + attenuated_excess
    
    # Perform spatial alpha-blending
    M_blob_orig = M_base[blob_mask]
    M_mod[blob_mask] = (1.0 - W_b) * M_blob_orig + W_b * M_fully_optimized_blob
    
    # --- Loss 1: Mean PSD Target Alignment ---
    current_psd = M_mod.mean(axis=1)
    psd_loss = np.sum((current_psd - target_psd) ** 2)
    
    # --- Loss 2: Local Range-Normalized SSIM Structure Guard ---
    candidate_blob = M_mod[blob_mask]
    orig_blob_norm = (M_blob_orig - M_blob_orig.min()) / (M_blob_orig.max() - M_blob_orig.min() + 1e-9)
    cand_blob_norm = (candidate_blob - candidate_blob.min()) / (candidate_blob.max() - candidate_blob.min() + 1e-9)
    relative_ssim_loss = 1.0 - ssim(orig_blob_norm, cand_blob_norm, data_range=1.0, win_size=7)
    
    return psd_loss + (1.0 * relative_ssim_loss)

# =====================================================================
# 4. RUN OPTIMIZER 
# =====================================================================
initial_gamma = [0.0]
bounds = [(0.0, 20.0)]

print("Optimizing cumulative transformation locked to a single threshold floor...")
res = minimize(
    scalar_threshold_objective,
    initial_gamma,
    args=(M, mask, rank_percentiles, W_blob, threshold_val, original_excess_scalar, reference_psd),
    bounds=bounds,
    method='L-BFGS-B',
    options={'ftol': 1e-12}
)

best_gamma = res.x[0]
print(f"Optimization Status: {res.success} | Optimal Attenuation Factor (Gamma): {best_gamma:.4f}")

# =====================================================================
# 5. EXECUTE FINAL RECONSTRUCTION
# =====================================================================
M_final_scalar = M.copy()
final_multiplier = np.power(rank_percentiles, best_gamma)
M_fully_optimized_blob = threshold_val + (original_excess_scalar * final_multiplier)

# Reapply spatial edge blending map
M_final_scalar[mask] = (1.0 - W_blob) * M[mask] + W_blob * M_fully_optimized_blob

# =====================================================================
# 6. VISUALIZE COMPARISON
# =====================================================================
fig, axes = plt.subplots(3, 1, figsize=(12, 10))

im0 = axes[0].pcolormesh(M, shading='auto', cmap='jet')
axes[0].set_title(f"Original Log2 M (Max: {M.max():.2f})")
fig.colorbar(im0, ax=axes[0])

im1 = axes[1].pcolormesh(M_final_scalar, shading='auto', cmap='jet', vmin=M_final_scalar.min(), vmax=M_final_scalar.max())
axes[1].set_title(f"Single Scalar Floor Attenuated Spectrogram (Floor: {threshold_val:.2f})")
fig.colorbar(im1, ax=axes[1])

axes[2].plot(f_M, original_psd, label='PSD original spectrogram', color='red', alpha=0.6)
axes[2].plot(f_M, reference_psd, label='Whittaker Target Baseline', color='black', linestyle='--')
axes[2].plot(f_M, np.mean(M_final_scalar, axis=1), label='PSD clean (Single Threshold Model)', color='orange', linewidth=2)
axes[2].set_title("PSD Profile Verification (Notice the tracking limit discrepancies)")
axes[2].legend()
axes[2].grid(True)

plt.tight_layout()
plt.show()

ValueError: zero-size array to reduction operation maximum which has no identity

No masl

In [213]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import minimize
from scipy.linalg import solve
from skimage.metrics import structural_similarity as ssim

# =====================================================================
# 1. WHITTAKER ASYMMETRIC LEAST SQUARES (Baseline Target Profile)
# =====================================================================
def whittaker_als_baseline(y, lam=1e4, p=0.01, max_iter=15):
    L = len(y)
    D = np.zeros((L-2, L))
    for i in range(L-2):
        D[i, i], D[i, i+1], D[i, i+2] = 1, -2, 1
    penalty = lam * np.dot(D.T, D)
    w = np.ones(L)
    for _ in range(max_iter):
        W = np.diag(w)
        z = solve(W + penalty, w * y)
        w = np.where(y > z, p, 1 - p)
    return z

# =====================================================================
# 2. SETUP DATA AND FULL 2D BASELINE MATRICES
# =====================================================================
# Assumes your full 2D Log2 matrix 'M' is loaded.
freq_bins, time_bins = M.shape
original_psd = M.mean(axis=1)
reference_psd = whittaker_als_baseline(original_psd, lam=1e4, p=0.01)

# Broadcast the 1D target baseline vector into a full 2D matrix matching M
# Every column now has the exact target floor for its frequency row
target_matrix_floor = np.tile(reference_psd[:, np.newaxis], (1, time_bins))

# Compute global log-excess above the baseline (clipping at 0 targets only positive peaks)
original_excess_global = np.clip(M - target_matrix_floor, 0.0, None)

# --- Full 2D Empirical Cumulative Distribution Function (eCDF) ---
# Flatten to calculate true global statistical ranks, then reshape back to 2D
M_flat = M.flatten()
sort_indices = np.argsort(M_flat)
rank_percentiles_flat = np.zeros_like(M_flat)
rank_percentiles_flat[sort_indices] = np.linspace(0.0, 1.0, len(M_flat))
rank_percentiles_2d = rank_percentiles_flat.reshape(freq_bins, time_bins)

# =====================================================================
# 3. OBJECTIVE FUNCTION FOR HIGH-PEAK COMPRESSION (Inverse Power-Law)
# =====================================================================
def global_spectrogram_objective(params, M_base, rank_pct_2d, target_floor_2d, original_excess_2d, target_psd):
    """
    Applies an inverse power-law transform to the eCDF.
    Selectively targets high-intensity peaks for attenuation while protecting background.
    """
    gamma = params[0]
    
    # Inverse power-law warp: 
    # Valid range for gamma is 0.0 to 1.0. Lower values compress peaks more aggressively.
    attenuation_multiplier = 1.0 - np.power(1.0 - rank_pct_2d, gamma)
    attenuated_excess = original_excess_2d * attenuation_multiplier
    
    # Reconstruct the candidate spectrogram matrix
    M_mod = target_floor_2d + attenuated_excess
    
    # --- Loss 1: Mean PSD Target Alignment ---
    current_psd = M_mod.mean(axis=1)
    psd_loss = np.sum((current_psd - target_psd) ** 2)
    
    # --- Loss 2: Global Native 2D SSIM Structure Guard ---
    orig_norm = (M_base - M_base.min()) / (M_base.max() - M_base.min() + 1e-9)
    mod_norm = (M_mod - M_mod.min()) / (M_mod.max() - M_mod.min() + 1e-9)
    global_ssim_loss = 1.0 - ssim(orig_norm, mod_norm, data_range=1.0)
    
    return psd_loss + (100.0 * global_ssim_loss)

# =====================================================================
# 4. RUN OPTIMIZER (Gamma bound between 0.0 and 1.0)
# =====================================================================
# Initial guess: gamma = 1.0 (Identity matrix, 100% untouched)
initial_gamma = [1.0]
# Lowering gamma toward 0.0 pinches the high-percentile peaks down
bounds = [(0.01, 1.0)] 

print("Optimizing global unmasked peak attenuation curve...")
res = minimize(
    global_spectrogram_objective,
    initial_gamma,
    args=(M, rank_percentiles_2d, target_matrix_floor, original_excess_global, reference_psd),
    bounds=bounds,
    method='L-BFGS-B',
    options={'ftol': 1e-12}
)

best_gamma = res.x[0]
print(f"Optimization Status: {res.success} | Optimal Peak Gamma: {best_gamma:.4f}")

# =====================================================================
# 5. EXECUTE FINAL GLOBAL RECONSTRUCTION
# =====================================================================
final_multiplier_2d = 1.0 - np.power(1.0 - rank_percentiles_2d, best_gamma)
M_final_global = target_matrix_floor + (original_excess_global * final_multiplier_2d)

# =====================================================================
# 6. VISUALIZE
# =====================================================================
fig, axes = plt.subplots(3, 1, figsize=(12, 10))

im0 = axes[0].pcolormesh(M, shading='auto', cmap='jet', vmin=M.min(), vmax=M.max())
axes[0].set_title(f"Original Full Spectrogram Matrix (Max: {M.max():.2f})")
fig.colorbar(im0, ax=axes[0])

im1 = axes[1].pcolormesh(M_final_global, shading='auto', cmap='jet')#, vmin=M.min(), vmax=M.max())
axes[1].set_title("Globally Attenuated Maskless Spectrogram Matrix")
fig.colorbar(im1, ax=axes[1])

axes[2].plot(f_M, original_psd, label='PSD original spectrogram', color='red', alpha=0.6)
axes[2].plot(f_M, reference_psd, label='Whittaker Target Baseline', color='black', linestyle='--')
axes[2].plot(f_M, np.mean(M_final_global, axis=1), label='PSD clean (Global Unmasked Model)', color='green', linewidth=2)
axes[2].set_title("Global PSD Profile Alignment Verification")
axes[2].legend()
axes[2].grid(True)

plt.tight_layout()
plt.show()

Optimizing global unmasked peak attenuation curve...
Optimization Status: True | Optimal Peak Gamma: 0.1555


In [212]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import minimize
from scipy.linalg import solve
from scipy.stats import beta
from skimage.metrics import structural_similarity as ssim

# =====================================================================
# 1. WHITTAKER ASYMMETRIC LEAST SQUARES (Baseline Target Profile)
# =====================================================================
def whittaker_als_baseline(y, lam=1e4, p=0.01, max_iter=15):
    """
    Computes a smooth baseline profile across the frequency spectrum.
    Bypasses high-intensity transient peaks to uncover the true background floor.
    """
    L = len(y)
    D = np.zeros((L-2, L))
    for i in range(L-2):
        D[i, i], D[i, i+1], D[i, i+2] = 1, -2, 1
    penalty = lam * np.dot(D.T, D)
    w = np.ones(L)
    for _ in range(max_iter):
        W = np.diag(w)
        z = solve(W + penalty, w * y)
        w = np.where(y > z, p, 1 - p)
    return z

# =====================================================================
# 2. SETUP DATA AND FULL 2D BASELINE MATRICES
# =====================================================================
# Assumes your full 2D Log2 matrix 'M' is loaded.
freq_bins, time_bins = M.shape
original_psd = M.mean(axis=1)
reference_psd = whittaker_als_baseline(original_psd, lam=1e4, p=0.01)

# Broadcast the 1D target baseline vector into a full 2D matrix matching M
# Every column now has the exact target floor for its frequency row
target_matrix_floor = np.tile(reference_psd[:, np.newaxis], (1, time_bins))

# Compute global log-excess above the baseline (clipping at 0 targets only positive peaks)
original_excess_global = np.clip(M - target_matrix_floor, 0.0, None)

# --- Full 2D Empirical Cumulative Distribution Function (eCDF) ---
# Flatten to calculate true global statistical ranks, then reshape back to 2D
M_flat = M.flatten()
sort_indices = np.argsort(M_flat)
rank_percentiles_flat = np.zeros_like(M_flat)
rank_percentiles_flat[sort_indices] = np.linspace(0.0, 1.0, len(M_flat))
rank_percentiles_2d = rank_percentiles_flat.reshape(freq_bins, time_bins)

# =====================================================================
# 3. OBJECTIVE FUNCTION: TWO-PARAMETER BETA CDF TRANSFORMATION
# =====================================================================
def global_spectrogram_objective(params, M_base, rank_pct_2d, target_floor_2d, original_excess_2d, target_psd):
    """
    Applies a two-parameter Beta CDF to the eCDF mapping across the entire spectrogram.
    Provides precise, independent control over peak suppression and background protection.
    """
    a_param, b_param = params[0], params[1]
    
    # Smooth, flexible, strictly ordered transformation map via Beta CDF
    # This acts as our attenuation multiplier matrix
    attenuation_multiplier = beta.cdf(rank_pct_2d, a=a_param, b=b_param)
    attenuated_excess = original_excess_2d * attenuation_multiplier
    
    # Reconstruct the candidate modified spectrogram matrix
    M_mod = target_floor_2d + attenuated_excess
    
    # --- Loss 1: Mean PSD Target Alignment ---
    current_psd = M_mod.mean(axis=1)
    psd_loss = np.sum((current_psd - target_psd) ** 2)
    
    # --- Loss 2: Global Native 2D SSIM Structure Guard ---
    # Min-max normalization maps the entire image matrix to [0, 1] to remove variance bias
    orig_norm = (M_base - M_base.min()) / (M_base.max() - M_base.min() + 1e-9)
    mod_norm = (M_mod - M_mod.min()) / (M_mod.max() - M_mod.min() + 1e-9)
    
    global_ssim = ssim(orig_norm, mod_norm, data_range=1.0)
    global_ssim_loss = 1.0 - global_ssim

    print(f'psd_loss: {psd_loss} | global_ssim_loss: {global_ssim_loss}')
    
    # Combine losses (1.0 weight balances shape matching vs energy reduction goals)
    return psd_loss + (100.0 * global_ssim_loss)

# =====================================================================
# 4. RUN OPTIMIZER (Two Parameters: Alpha and Beta)
# =====================================================================
# Initial guess: a=1.0, b=1.0 (Yields a completely linear identity transform, untouched)
initial_params = [1.0, 1.0]

# Bounds configuration: 
# Setting lower bound of 'a' to 1.0 prevents the background from being squashed prematurely.
# Allowing 'b' to stretch gives the curve freedom to shave off the top peaks cleanly.
bounds = [(1.0, 3.0), (0.5, 2.0)]

print("Optimizing dual-parameter Beta cumulative transformation matrix globally...")
res = minimize(
    global_spectrogram_objective,
    initial_params,
    args=(M, rank_percentiles_2d, target_matrix_floor, original_excess_global, reference_psd),
    bounds=bounds,
    method='L-BFGS-B',
    options={'ftol': 1e-12}
)

best_a, best_b = res.x[0], res.x[1]
print(f"Optimization Status: {res.success}")
print(f"Optimal Alpha (a): {best_a:.4f} | Optimal Beta (b): {best_b:.4f}")

# =====================================================================
# 5. EXECUTE FINAL GLOBAL RECONSTRUCTION
# =====================================================================
final_multiplier_2d = beta.cdf(rank_percentiles_2d, a=best_a, b=best_b)
M_final_global = target_matrix_floor + (original_excess_global * final_multiplier_2d)

print("Processing complete.")

# =====================================================================
# 6. VISUALIZE RESULTS
# =====================================================================
fig, axes = plt.subplots(3, 1, figsize=(12, 10))

im0 = axes[0].pcolormesh(M, shading='auto', cmap='jet')
axes[0].set_title(f"Original Full Spectrogram Matrix (Max: {M.max():.2f})")
fig.colorbar(im0, ax=axes[0])

im1 = axes[1].pcolormesh(M_final_global, shading='auto', cmap='jet')#, vmin=M_final_global.min(), vmax=M_final_global.max())
axes[1].set_title("Globally Cleaned Spectrogram (Beta CDF Warp)")
fig.colorbar(im1, ax=axes[1])

axes[2].plot(f_M, original_psd, label='PSD original spectrogram', color='red', alpha=0.6)
axes[2].plot(f_M, reference_psd, label='Whittaker Target Baseline', color='black', linestyle='--')
axes[2].plot(f_M, np.mean(M_final_global, axis=1), label='PSD clean (Dual-Parameter Model)', color='green', linewidth=2)
axes[2].set_title("Global PSD Profile Alignment Verification")
axes[2].legend()
axes[2].grid(True)

plt.tight_layout()
plt.show()

Optimizing dual-parameter Beta cumulative transformation matrix globally...
psd_loss: 68.61372692374572 | global_ssim_loss: 0.5224907728666198
psd_loss: 68.61372664809635 | global_ssim_loss: 0.5224907736122097
psd_loss: 68.61372735523226 | global_ssim_loss: 0.522490772219097
psd_loss: 15.517937664420947 | global_ssim_loss: 0.6928095569380345
psd_loss: 15.51793771553927 | global_ssim_loss: 0.6928095565530923
psd_loss: 15.517938095882835 | global_ssim_loss: 0.6928095551660782
Optimization Status: True
Optimal Alpha (a): 3.0000 | Optimal Beta (b): 0.5000
Processing complete.


clean

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import minimize
from scipy.linalg import solve
from scipy.stats import beta
from scipy.ndimage import distance_transform_edt
from skimage.metrics import structural_similarity as ssim

# =====================================================================
# 1. WHITTAKER ASYMMETRIC LEAST SQUARES (Baseline Target Profile)
# =====================================================================
def whittaker_als_baseline(y, lam=1e4, p=0.01, max_iter=15):
    """
    Computes a smooth baseline profile across the frequency spectrum.
    Bypasses high-intensity transient peaks to uncover the true background floor.
    """
    L = len(y)
    D = np.zeros((L-2, L))
    for i in range(L-2):
        D[i, i], D[i, i+1], D[i, i+2] = 1, -2, 1
    penalty = lam * np.dot(D.T, D)
    w = np.ones(L)
    for _ in range(max_iter):
        W = np.diag(w)
        z = solve(W + penalty, w * y)
        w = np.where(y > z, p, 1 - p)
    return z


# =====================================================================
# 2. DEFINING THE MASK AND ISOLATING LOG PROPERTIES
# =====================================================================
# Assuming your 2D Log2 matrix 'M' is already loaded in your environment.
freq_bins, time_bins = M.shape
f_M = np.arange(freq_bins) # Frequency axis for plotting

original_psd = M.mean(axis=1)
reference_psd = whittaker_als_baseline(original_psd, lam=1e4, p=0.01)

# --- THRESHOLDING SETUP ---
# Select a threshold value to isolate your high-amplitude peaks.
# Modify this value (or use a percentile like np.percentile(M, 90)) to fit your data.
threshold_val = np.quantile(M, 0.75)
mask = M > threshold_val

# Extract the elements inside the localized mask region
M_blob_orig = M[mask]

# Calculate the Empirical Cumulative Distribution Function (eCDF) inside the mask
# This maps the relative hierarchy inside the mask onto a perfect [0, 1] scale
sort_indices = np.argsort(M_blob_orig)
rank_percentiles = np.zeros_like(M_blob_orig)
rank_percentiles[sort_indices] = np.linspace(0.0, 1.0, len(M_blob_orig))

# Calculate the log-excess energy strictly relative to your chosen threshold floor
original_excess = np.clip(M_blob_orig - threshold_val, 0.0, None)

# --- SPATIAL BOUNDARY SMOOTHING MAP ---
distance_map = distance_transform_edt(mask)
transition_width = 6.0  # Adjust width of the alpha-blending zone at mask edges
weights = 1.0 - np.exp(-(distance_map / transition_width) ** 2)
weights[~mask] = 0.0
W_blob = weights[mask]

# =====================================================================
# 3. OBJECTIVE FUNCTION: LOCALIZED PARAMETERIZED TRANSFORMATION
# =====================================================================
def masked_attenuation_objective(params, M_base, blob_mask, rank_pct, W_b, threshold_val, original_excess, target_psd):
    """
    Optimizes Beta CDF parameters to warp the mask's eCDF. Attenuates log-excess
    energy down toward the target PSD while forcing a hard floor constraint at threshold_val.
    """
    a_param, b_param = params[0], params[1]
    
    M_mod = M_base.copy()
    
    # 1. Parameterized monotonic transform of the cumulative distribution
    attenuation_multiplier = beta.cdf(rank_pct, a=a_param, b=b_param)
    
    # 2. Attenuate the excess envelope
    attenuated_excess = original_excess * attenuation_multiplier
    
    # 3. Reconstruct. Since attenuated_excess >= 0, values can NEVER go below threshold_val
    M_fully_optimized_blob = threshold_val + attenuated_excess
    
    # 4. Perform localized spatial alpha-blending to eliminate boundary seams
    M_blob_orig = M_base[blob_mask]
    M_mod[blob_mask] = (1.0 - W_b) * M_blob_orig + W_b * M_fully_optimized_blob
    
    # --- Loss Term 1: Mean PSD Alignment Error ---
    current_psd = M_mod.mean(axis=1)
    psd_loss = np.mean((current_psd - target_psd) ** 2)
    
    # --- Loss Term 2: Range-Normalized Relative SSIM Guard ---
    candidate_blob = M_mod[blob_mask]
    orig_blob_norm = (M_blob_orig - M_blob_orig.min()) / (M_blob_orig.max() - M_blob_orig.min() + 1e-9)
    cand_blob_norm = (candidate_blob - candidate_blob.min()) / (candidate_blob.max() - candidate_blob.min() + 1e-9)
    relative_ssim_loss = 1.0 - ssim(orig_blob_norm, cand_blob_norm, data_range=1.0, win_size=7)
    
    print(psd_loss, relative_ssim_loss)

    # Balanced loss weighting to prevent structural collapse
    return 10000*psd_loss + (10.0 * relative_ssim_loss)

# =====================================================================
# 4. RUN PARAMETRIC OPTIMIZATION
# =====================================================================
# Initial guess: a=1.0, b=1.0 (Corresponds to completely untouched linear state)
initial_params = [1.0, 1.0]

# Bounds configuration:
# Restricting a >= 1.0 and b <= 1.0 creates a curve shape that suppresses high-rank 
# peak pixels while leaving low-rank boundary values safely near their baseline values.
bounds = [(0.01, 1.0), (1.0, 50.0)]

print("Optimizing masked cumulative transfer parameters under structural constraints...")
res = minimize(
    masked_attenuation_objective,
    initial_params,
    args=(M, mask, rank_percentiles, W_blob, threshold_val, original_excess, reference_psd),
    bounds=bounds,
    method='L-BFGS-B',
    options={'ftol': 1e-12}
)

best_a, best_b = res.x[0], res.x[1]
print(f"Optimization Status: {res.success}")
print(f"Optimal Alpha (a): {best_a:.4f} | Optimal Beta (b): {best_b:.4f}")

# =====================================================================
# 5. EXECUTE FINAL COMPOSITION
# =====================================================================
M_final = M.copy()
final_multiplier = beta.cdf(rank_percentiles, a=best_a, b=best_b)
M_fully_optimized_blob = threshold_val + (original_excess * final_multiplier)

# Merge back cleanly via our boundary alpha-weights matrix
M_final[mask] = (1.0 - W_blob) * M[mask] + W_blob * M_fully_optimized_blob

# =====================================================================
# 6. VISUAL COMPONENT VERIFICATION
# =====================================================================
fig, axes = plt.subplots(3, 1, figsize=(12, 10))

im0 = axes[0].pcolormesh(M, shading='auto', cmap='plasma')
axes[0].set_title(f"Original Spectrogram (Max Peak: {M.max():.2f})")
fig.colorbar(im0, ax=axes[0])

im1 = axes[1].pcolormesh(M_final, shading='auto', cmap='plasma', vmin=M_final.min(), vmax=M_final.max())
axes[1].set_title(f"Masked Bounded-Attenuated Spectrogram (Hard Floor Limit: {threshold_val:.2f})")
fig.colorbar(im1, ax=axes[1])

axes[2].plot(f_M, original_psd, label='PSD original spectrogram', color='red', alpha=0.6)
axes[2].plot(f_M, reference_psd, label='Whittaker Target Baseline', color='black', linestyle='--')
axes[2].plot(f_M, np.mean(M_final, axis=1), label='PSD cleaned spectrogram', color='green', linewidth=2)
axes[2].axhline(threshold_val, color='blue', linestyle=':', label='Hard Mask Floor Threshold')
axes[2].set_title("PSD Attenuation Analysis")
axes[2].legend()
axes[2].grid(True)

plt.tight_layout()
plt.show()

Optimizing masked cumulative transfer parameters under structural constraints...


ValueError: zero-size array to reduction operation minimum which has no identity

In [269]:
M = spectro.copy()
mask_f = f_spectro >= 20
f_M = f_spectro[mask_f]
M = M[mask_f, :]
#M = np.log2(M + 1e-11)

fig, axis = plt.subplots(1)
axis.pcolormesh(t_spectro, f_M, M, shading = 'nearest', cmap = 'jet')

mask = M >= np.quantile(M, 0.75)


In [270]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import minimize
from scipy.linalg import solve
from scipy.stats import beta
from scipy.ndimage import distance_transform_edt
from skimage.metrics import structural_similarity as ssim

# =====================================================================
# 1. WHITTAKER ASYMMETRIC LEAST SQUARES (Baseline Target Profile)
# =====================================================================
def whittaker_als_baseline(y, lam=1e4, p=0.01, max_iter=15):
    L = len(y)
    D = np.zeros((L-2, L))
    for i in range(L-2):
        D[i, i], D[i, i+1], D[i, i+2] = 1, -2, 1
    penalty = lam * np.dot(D.T, D)
    w = np.ones(L)
    for _ in range(max_iter):
        W = np.diag(w)
        z = solve(W + penalty, w * y)
        w = np.where(y > z, p, 1 - p)
    return z

# =====================================================================
# 2. SETUP DATA AND MASK PROPERTIES (Scale-Agnostic Linear Framework)
# =====================================================================
freq_bins, time_bins = M.shape
f_M = np.arange(freq_bins)

original_psd = M.mean(axis=1)
reference_psd = whittaker_als_baseline(original_psd, lam=1e4, p=0.01)

M_blob_orig = M[mask]

# --- Extract Scale Normalization Factor ---
# Finding the maximum value inside the mask to prevent underflow issues
blob_max = M_blob_orig.max()
if blob_max == 0:
    blob_max = 1.0 

# --- 1D Empirical Cumulative Distribution Function (eCDF) ---
sort_indices = np.argsort(M_blob_orig)
rank_percentiles = np.zeros_like(M_blob_orig)
rank_percentiles[sort_indices] = np.linspace(0.0, 1.0, len(M_blob_orig))

# --- Spatial Blending Weights Map ---
distance_map = distance_transform_edt(mask)
transition_width = 1.0 
weights = 1.0 - np.exp(-(distance_map / transition_width) ** 2)
weights[~mask] = 0.0
W_blob = weights[mask]

# =====================================================================
# 3. OBJECTIVE FUNCTION: SCALE-NORMALIZED LOSS RUNNER
# =====================================================================
def normalized_linear_objective(params, M_base, blob_mask, rank_pct, W_b, target_psd, scale_factor):
    a_param, b_param = params[0], params[1]
    M_mod = M_base.copy()
    M_blob_orig = M_base[blob_mask]
    
    # 1. Monotonic percentile warping multiplier
    attenuation_multiplier = beta.cdf(rank_pct, a=a_param, b=b_param)
    
    # 2. Direct linear scaling
    M_fully_optimized_blob = M_blob_orig * attenuation_multiplier
    M_mod[blob_mask] = (1.0 - W_b) * M_blob_orig + W_b * M_fully_optimized_blob
    
    # --- CRITICAL STEP: Scale Up to Prevent Underflow ---
    # We divide by scale_factor so the values land nicely near 1.0 during the subtraction
    current_psd_norm = M_mod.mean(axis=1) / scale_factor
    target_psd_norm = target_psd / scale_factor
    
    psd_loss = np.mean((current_psd_norm - target_psd_norm) ** 2)
    
    # --- Loss 2: Local Range-Normalized SSIM Structure Guard ---
    candidate_blob = M_mod[blob_mask]
    orig_blob_norm = (M_blob_orig - M_blob_orig.min()) / (M_blob_orig.max() - M_blob_orig.min() + 1e-9)
    cand_blob_norm = (candidate_blob - candidate_blob.min()) / (candidate_blob.max() - candidate_blob.min() + 1e-9)
    relative_ssim_loss = 1.0 - ssim(orig_blob_norm, cand_blob_norm, data_range=1.0, win_size=7)
    
    # Give the normalized PSD loss a strong vote
    return (10.0 * psd_loss) + relative_ssim_loss

# =====================================================================
# 4. RUN OPTIMIZER (Now fully sensitive to changes)
# =====================================================================
initial_params = [1.0, 1.0]
bounds = [(0.01, 1.0), (1.0, 100.0)]

print("Optimizing scale-normalized linear parameters...")
res = minimize(
    normalized_linear_objective,
    initial_params,
    args=(M, mask, rank_percentiles, W_blob, reference_psd, blob_max),
    bounds=bounds,
    method='L-BFGS-B',
    options={'ftol': 1e-15, 'gtol': 1e-15} # Pushed tolerances down to intercept small variations
)

best_a, best_b = res.x[0], res.x[1]
print(f"Optimization Status: {res.success} | Optimal a: {best_a:.4f}, b: {best_b:.4f}")

# =====================================================================
# 5. EXECUTE FINAL RECONSTRUCTION
# =====================================================================
M_final = M.copy()
final_multiplier = beta.cdf(rank_percentiles, a=best_a, b=best_b)

# Apply the validated filter curve straight to the original tiny scale values
M_fully_optimized_blob = M[mask] * final_multiplier
M_final[mask] = (1.0 - W_blob) * M[mask] + W_blob * M_fully_optimized_blob

# =====================================================================
# 5. EXECUTE FINAL RECONSTRUCTION
# =====================================================================
M_final = M.copy()
final_multiplier = beta.cdf(rank_percentiles, a=best_a, b=best_b)

# Direct linear scaling
M_fully_optimized_blob = M[mask] * final_multiplier

# Blend back seamlessly using the boundary weights map
M_final[mask] = (1.0 - W_blob) * M[mask] + W_blob * M_fully_optimized_blob

# =====================================================================
# 6. VISUALIZE RESULTS (Linear-Domain Plots)
# =====================================================================
fig, axes = plt.subplots(4, 1, figsize=(12, 12))

# Panel 1: Original Raw Linear Spectrogram
im0 = axes[0].pcolormesh(M, shading='auto', cmap='plasma')
axes[0].set_title(f"Original Linear Spectrogram Matrix M (Max Peak: {M.max():.2f})")
axes[0].set_ylabel("Frequency Bins")
fig.colorbar(im0, ax=axes[0], label="Intensity (Linear Power)")

# Panel 2: Cleaned Order-Preserved Linear Spectrogram
im1 = axes[1].pcolormesh(M_final, shading='auto', cmap='plasma', vmin=M.min(), vmax=M.max())
axes[1].set_title("Cleaned Spectrogram (Pure Linear Domain Attenuation)")
axes[1].set_ylabel("Frequency Bins")
fig.colorbar(im1, ax=axes[1], label="Intensity (Linear Power)")

# Panel 3: Linear PSD Curve Comparison Validation
axes[2].plot(f_M, original_psd, label='PSD Original Spectrogram', color='red', alpha=0.6)
axes[2].plot(f_M, reference_psd, label='Whittaker Target Baseline', color='black', linestyle='--')
axes[2].plot(f_M, np.mean(M_final, axis=1), label='PSD Cleaned Spectrogram (Pure Linear)', color='green', linewidth=2)
axes[2].set_title("PSD Profile Verification Analysis (Linear Domain)")
axes[2].set_xlabel("Frequency Bins")
axes[2].set_ylabel("Mean Linear Intensity")
axes[2].legend()
axes[2].grid(True)

axes[3].pcolormesh(M_final-M, shading='auto', cmap='plasma')

plt.tight_layout()
plt.show()

Optimizing scale-normalized linear parameters...
Optimization Status: False | Optimal a: 0.6135, b: 1.0000


In [248]:
fig, axes = plt.subplots(3, sharex = True, sharey = True, constrained_layout = True)
axes[0].pcolormesh(t_spectro, f_M, M, shading = 'nearest', cmap = 'jet')
axes[0].contour(t_spectro, f_M, mask, colors = 'white', linewidths=2)
axes[1].pcolormesh(t_spectro, f_M, M_smooth_clean, shading = 'nearest', cmap = 'jet')
axes[1].contour(t_spectro, f_M, mask, colors = 'white', linewidths=2)
axes[2].pcolormesh(t_spectro, f_M, mask, shading = 'nearest', cmap = 'jet')

plt.show()

In [299]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.linalg import solve
from scipy.ndimage import distance_transform_edt

# =====================================================================
# 1. WHITTAKER ASYMMETRIC LEAST SQUARES (Baseline Target Profile)
# =====================================================================
def whittaker_als_baseline(y, lam=1e4, p=0.01, max_iter=15):
    """
    Computes a smooth baseline profile across the frequency spectrum.
    Bypasses high-intensity transient peaks to uncover the true background floor.
    """
    L = len(y)
    D = np.zeros((L-2, L))
    for i in range(L-2):
        D[i, i], D[i, i+1], D[i, i+2] = 1, -2, 1
    penalty = lam * np.dot(D.T, D)
    w = np.ones(L)
    for _ in range(max_iter):
        W = np.diag(w)
        z = solve(W + penalty, w * y)
        w = np.where(y > z, p, 1 - p)
    return z

# =====================================================================
# 2. SETUP DATA AND GLOBAL MASKING
# =====================================================================
# Assumes your 2D RAW LINEAR matrix 'M' is loaded.
freq_bins, time_bins = M.shape
f_M = np.arange(freq_bins)

original_psd = M.mean(axis=1)
reference_psd = whittaker_als_baseline(original_psd, lam=1e4, p=0.01)

# Set Threshold T (e.g., 90th percentile of the full data)
T = np.percentile(M, 50)
mask = M > T

# Isolate the data inside the active mask
M_blob_orig = M[mask]

# --- Spatial Blending Weights Map ---
# Smooths the edges of the mask to prevent blocky, pixelated visual seams
distance_map = distance_transform_edt(mask)
transition_width = 1.0 
weights = 1.0 - np.exp(-(distance_map / transition_width) ** 2)
weights[~mask] = 0.0
W_blob = weights[mask]

# =====================================================================
# 3. ANALYTIC SPECIFICATION OF TARGET CDF (f1) & QUANTILE MATCHING
# =====================================================================
# Calculate the empirical probability of being below or equal to T
prob_at_T = np.sum(M <= T) / M.size

# Define the absolute boundaries for f1 above the threshold
min_x_above = T
max_x_above = T * 3.0

# Generate an array of target intensities above T to construct f1
x_grid_above = np.linspace(min_x_above, max_x_above, 2000)

# Build the smooth concave bridge for f1 (using gamma=0.5 for square-root concavity)
gamma = 0.5 
f1_probabilities_above = prob_at_T + (1.0 - prob_at_T) * np.power((x_grid_above - min_x_above) / (max_x_above - min_x_above), gamma)

# Combine below and above definitions to establish a complete global target coordinate map
# Below T, f1 matches the sorted values of M that are <= T
M_below_T = np.sort(M[M <= T])
f1_x_below = M_below_T
f1_y_below = np.linspace(0.0, prob_at_T, len(M_below_T))

# Complete global coordinates for the f1 curve definition
f1_x_curve = np.concatenate([f1_x_below, x_grid_above])
f1_y_curve = np.concatenate([f1_y_below, f1_probabilities_above])

# --- QUANTILE MATCHING OPERATION ---
# For every pixel inside the mask, find its original empirical percentile rank in f0
sorted_all_M = np.sort(M.flatten())
ranks_in_f0 = np.searchsorted(sorted_all_M, M_blob_orig) / len(sorted_all_M)

# Map these exact ranks through the inverse of our new target curve f1_inverse(prob)
M_quantile_matched_blob = np.interp(ranks_in_f0, f1_y_curve, f1_x_curve)

# =====================================================================
# 4. FINAL RECONSTRUCTION WITH FIXED UPPER LIMIT GUARD
# =====================================================================
M_final = M.copy()

# 1. Smoothly blend the mapped quantile values at the spatial boundaries
M_final[mask] = (1.0 - W_blob) * M[mask] + W_blob * M_quantile_matched_blob

# 2. FIXED UPPER BOUNDARY LIMIT GUARD
# Enforce the clipping explicitly on the mask AFTER blending.
# This prevents uncorrected pixel values from bleeding back past the 2T limit.
M_final[mask] = np.clip(M_final[mask], None, max_x_above)

M_difference = M - M_final

# =====================================================================
# 5. VISUALIZATION (SPECTROGRAMS, PSDS, AND TRULY SMOOTH OVERLAID CDFs)
# =====================================================================
fig, axes = plt.subplots(5, 1, figsize=(12, 24))

# Panel 1: Original Spectrogram
im0 = axes[0].pcolormesh(M, shading='auto', cmap='jet')
axes[0].set_title(f"Original Linear Spectrogram Matrix M (True Max: {M.max():.2e})")
axes[0].set_ylabel("Frequency Bins")
fig.colorbar(im0, ax=axes[0], label="Linear Power")

# Panel 2: Quantile-Matched Spectrogram
im1 = axes[1].pcolormesh(M_final, shading='auto', cmap='jet', vmin=M.min(), vmax=M.max())
axes[1].set_title(f"Quantile-Matched Spectrogram (Guaranteed Max Ceiling: {M_final[mask].max():.2e})")
axes[1].set_ylabel("Frequency Bins")
fig.colorbar(im1, ax=axes[1], label="Linear Power")

# Panel 3: 2D Spectrogram Difference Map
im2 = axes[2].pcolormesh(M_difference, shading='auto', cmap='jet')
axes[2].set_title("2D Spectrogram Difference Map (Energy Removed Via Quantile Specification)")
axes[2].set_ylabel("Frequency Bins")
fig.colorbar(im2, ax=axes[2], label="Amplitude Reduced")

# Panel 4: Linear PSD Curve Comparison
axes[3].plot(f_M, original_psd, label='PSD Original Spectrogram', color='red', alpha=0.6)
axes[3].plot(f_M, reference_psd, label='Whittaker Target Baseline', color='black', linestyle='--')
axes[3].plot(f_M, np.mean(M_final, axis=1), label='PSD Cleaned Spectrogram', color='green', linewidth=2)
axes[3].set_title("PSD Profile Verification Analysis")
axes[3].set_ylabel("Mean Linear Intensity")
axes[3].legend()
axes[3].grid(True)

# Panel 5: GLOBAL INTENSITY CDF VISUALIZATION (With logarithmic x-scale)
sorted_M_orig = np.sort(M.flatten())
f0_y = np.linspace(0.0, 1.0, len(sorted_M_orig))

sorted_M_final = np.sort(M_final.flatten())
f_final_y = np.linspace(0.0, 1.0, len(sorted_M_final))

# Plot all distributions
axes[4].plot(sorted_M_orig, f0_y, label='Original CDF (f0)', color='red', alpha=0.5, linewidth=3)
axes[4].plot(f1_x_curve, f1_y_curve, label='Analytic Target CDF (f1 Specified)', color='blue', linestyle='--', linewidth=2)
axes[4].plot(sorted_M_final, f_final_y, label='Actual Resulting CDF (f_final)', color='green', linewidth=2)

# Label markers for constraints
axes[4].axvline(T, color='purple', linestyle=':', label=f'Threshold T ({T:.2e})')
axes[4].axvline(max_x_above, color='cyan', linestyle=':', label=f'Hard Ceiling Upper Limit 2T ({max_x_above:.2e})')

axes[4].set_title("Global Intensity Cumulative Distribution Function (CDF) Specification")
axes[4].set_xlabel("Pixel Amplitude Value (Log Scale to view the tail)")
axes[4].set_ylabel("Probability")
axes[4].legend()
axes[4].grid(True, which="both", ls="-")

plt.tight_layout()
plt.show()

In [300]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.linalg import solve
from scipy.ndimage import distance_transform_edt

# =====================================================================
# 1. WHITTAKER ASYMMETRIC LEAST SQUARES (Baseline Target Profile)
# =====================================================================
def whittaker_als_baseline(y, lam=1e4, p=0.01, max_iter=15):
    """
    Computes a smooth baseline profile across the frequency spectrum.
    Bypasses high-intensity transient peaks to uncover the true background floor.
    """
    L = len(y)
    D = np.zeros((L-2, L))
    for i in range(L-2):
        D[i, i], D[i, i+1], D[i, i+2] = 1, -2, 1
    penalty = lam * np.dot(D.T, D)
    w = np.ones(L)
    for _ in range(max_iter):
        W = np.diag(w)
        z = solve(W + penalty, w * y)
        w = np.where(y > z, p, 1 - p)
    return z

# =====================================================================
# 2. SETUP DATA AND GLOBAL MASKING
# =====================================================================
# Assumes your 2D RAW LINEAR matrix 'M' is loaded.
freq_bins, time_bins = M.shape
f_M = np.arange(freq_bins)

original_psd = M.mean(axis=1)
reference_psd = whittaker_als_baseline(original_psd, lam=1e4, p=0.01)

# Set Threshold T (e.g., 50th percentile of the full data)
T = np.percentile(M, 50)
mask = M > T

# Isolate the data inside the active mask
M_blob_orig = M[mask]

# --- Spatial Blending Weights Map ---
# Smooths the edges of the mask to prevent blocky, pixelated visual seams
distance_map = distance_transform_edt(mask)
transition_width = 1.0 
weights = 1.0 - np.exp(-(distance_map / transition_width) ** 2)
weights[~mask] = 0.0
W_blob = weights[mask]

# =====================================================================
# 3. ANALYTIC SPECIFICATION OF TARGET CDF (f1) & QUANTILE MATCHING
# =====================================================================
# Calculate the empirical probability of being below or equal to T
prob_at_T = np.sum(M <= T) / M.size

# Define the absolute boundaries for f1 above the threshold
min_x_above = T
max_x_above = T * 3.0

# Generate an array of target intensities above T to construct f1
x_grid_above = np.linspace(min_x_above, max_x_above, 2000)

# Build the smooth concave bridge for f1 (using gamma=0.5 for square-root concavity)
gamma = 0.5 
f1_probabilities_above = prob_at_T + (1.0 - prob_at_T) * np.power((x_grid_above - min_x_above) / (max_x_above - min_x_above), gamma)

# Combine below and above definitions to establish a complete global target coordinate map
# Below T, f1 matches the sorted values of M that are <= T
M_below_T = np.sort(M[M <= T])
f1_x_below = M_below_T
f1_y_below = np.linspace(0.0, prob_at_T, len(M_below_T))

# Complete global coordinates for the f1 curve definition
f1_x_curve = np.concatenate([f1_x_below, x_grid_above])
f1_y_curve = np.concatenate([f1_y_below, f1_probabilities_above])

# --- QUANTILE MATCHING OPERATION ---
# For every pixel inside the mask, find its original empirical percentile rank in f0
sorted_all_M = np.sort(M.flatten())
ranks_in_f0 = np.searchsorted(sorted_all_M, M_blob_orig) / len(sorted_all_M)

# Map these exact ranks through the inverse of our new target curve f1_inverse(prob)
M_quantile_matched_blob = np.interp(ranks_in_f0, f1_y_curve, f1_x_curve)

# =====================================================================
# 4. FINAL RECONSTRUCTION WITH FIXED UPPER LIMIT GUARD
# =====================================================================
M_final = M.copy()

# 1. Smoothly blend the mapped quantile values at the spatial boundaries
M_final[mask] = (1.0 - W_blob) * M[mask] + W_blob * M_quantile_matched_blob

# 2. FIXED UPPER BOUNDARY LIMIT GUARD
M_final[mask] = np.clip(M_final[mask], None, max_x_above)

M_difference = M - M_final

# =====================================================================
# 5. VISUALIZATION (SPECTROGRAMS, PSDS, AND POPULATION CDFs)
# =====================================================================
fig, axes = plt.subplots(6, 1, figsize=(12, 28))

# Panel 1: Original Spectrogram
im0 = axes[0].pcolormesh(M, shading='auto', cmap='jet')
axes[0].set_title(f"Original Linear Spectrogram Matrix M (True Max: {M.max():.2e})")
axes[0].set_ylabel("Frequency Bins")
fig.colorbar(im0, ax=axes[0], label="Linear Power")

# Panel 2: Quantile-Matched Spectrogram
im1 = axes[1].pcolormesh(M_final, shading='auto', cmap='jet', vmin=M.min(), vmax=M.max())
axes[1].set_title(f"Quantile-Matched Spectrogram (Guaranteed Max Ceiling: {M_final[mask].max():.2e})")
axes[1].set_ylabel("Frequency Bins")
fig.colorbar(im1, ax=axes[1], label="Linear Power")

# Panel 3: 2D Spectrogram Difference Map
im2 = axes[2].pcolormesh(M_difference, shading='auto', cmap='jet')
axes[2].set_title("2D Spectrogram Difference Map (Energy Removed Via Quantile Specification)")
axes[2].set_ylabel("Frequency Bins")
fig.colorbar(im2, ax=axes[2], label="Amplitude Reduced")

# Panel 4: Linear PSD Curve Comparison
axes[3].plot(f_M, original_psd, label='PSD Original Spectrogram', color='red', alpha=0.6)
axes[3].plot(f_M, reference_psd, label='Whittaker Target Baseline', color='black', linestyle='--')
axes[3].plot(f_M, np.mean(M_final, axis=1), label='PSD Cleaned Spectrogram', color='green', linewidth=2)
axes[3].set_title("PSD Profile Verification Analysis")
axes[3].set_ylabel("Mean Linear Intensity")
axes[3].legend()
axes[3].grid(True)

# Panel 5: GLOBAL INTENSITY CDF VISUALIZATION
sorted_M_orig = np.sort(M.flatten())
f0_y = np.linspace(0.0, 1.0, len(sorted_M_orig))

sorted_M_final = np.sort(M_final.flatten())
f_final_y = np.linspace(0.0, 1.0, len(sorted_M_final))

axes[4].plot(sorted_M_orig, f0_y, label='Original Global CDF (f0)', color='red', alpha=0.5, linewidth=3)
axes[4].plot(f1_x_curve, f1_y_curve, label='Analytic Target CDF (f1 Specified)', color='blue', linestyle='--', linewidth=2)
axes[4].plot(sorted_M_final, f_final_y, label='Actual Resulting Global CDF', color='green', linewidth=2)
axes[4].axvline(T, color='purple', linestyle=':', label=f'Threshold T ({T:.2e})')
axes[4].axvline(max_x_above, color='cyan', linestyle=':', label=f'Hard Ceiling Upper Limit 3T ({max_x_above:.2e})')
axes[4].set_title("Global Intensity Cumulative Distribution Function (CDF) Specification")
axes[4].set_ylabel("Probability")
axes[4].set_xscale('log')
axes[4].legend()
axes[4].grid(True, which="both", ls="-")

# Panel 6: SUB-POPULATION COHORT CDF (Selected vs. Unselected Pixels)
# Extract populations using the mask state from both matrices
unselected_pixels = np.sort(M[~mask])
selected_pixels_orig = np.sort(M_blob_orig)
selected_pixels_final = np.sort(M_final[mask])

# Generate local coordinate probability arrays normalized individually [0, 1]
y_unsel = np.linspace(0.0, 1.0, len(unselected_pixels))
y_sel_orig = np.linspace(0.0, 1.0, len(selected_pixels_orig))
y_sel_final = np.linspace(0.0, 1.0, len(selected_pixels_final))

# Plot subsets
axes[5].plot(unselected_pixels, y_unsel, label=f'Unselected Pixels (M <= T)', color='gray', linestyle='-', linewidth=2)
axes[5].plot(selected_pixels_orig, y_sel_orig, label=f'Selected Mask Pixels (Original Tail)', color='darkred', alpha=0.6, linewidth=2.5)
axes[5].plot(selected_pixels_final, y_sel_final, label=f'Selected Mask Pixels (Transformed Tail)', color='darkgreen', linewidth=2.5)

axes[5].axvline(T, color='purple', linestyle=':', label=f'Threshold T ({T:.2e})')
axes[5].axvline(max_x_above, color='cyan', linestyle=':', label=f'Max 3T Boundary ({max_x_above:.2e})')
axes[5].set_title("Isolated Sub-Population Empirical CDF Comparison")
axes[5].set_xlabel("Pixel Amplitude Value (Log Scale)")
axes[5].set_ylabel("Internal Cohort Probability")
axes[5].set_xscale('log')
axes[5].legend()
axes[5].grid(True, which="both", ls="-")

plt.tight_layout()
plt.show()

In [305]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.linalg import solve
from scipy.ndimage import distance_transform_edt

# =====================================================================
# 1. WHITTAKER ASYMMETRIC LEAST SQUARES (Baseline Target Profile)
# =====================================================================
def whittaker_als_baseline(y, lam=1e4, p=0.01, max_iter=15):
    """
    Computes a smooth baseline profile across the frequency spectrum.
    Bypasses high-intensity transient peaks to uncover the true background floor.
    """
    L = len(y)
    D = np.zeros((L-2, L))
    for i in range(L-2):
        D[i, i], D[i, i+1], D[i, i+2] = 1, -2, 1
    penalty = lam * np.dot(D.T, D)
    w = np.ones(L)
    for _ in range(max_iter):
        W = np.diag(w)
        z = solve(W + penalty, w * y)
        w = np.where(y > z, p, 1 - p)
    return z

# =====================================================================
# 2. SETUP DATA AND GLOBAL MASKING
# =====================================================================
# Assumes your 2D RAW LINEAR matrix 'M' is loaded.
freq_bins, time_bins = M.shape
f_M = np.arange(freq_bins)

original_psd = M.mean(axis=1)
reference_psd = whittaker_als_baseline(original_psd, lam=1e4, p=0.01)

# Set Threshold T (using 50th percentile as requested)
T = np.percentile(M, 50)
mask = M > T

# Isolate the data inside the active mask
M_blob_orig = M[mask]

# --- Spatial Blending Weights Map ---
distance_map = distance_transform_edt(mask)
transition_width = 0.0 
weights = 1.0 - np.exp(-(distance_map / transition_width) ** 2)
weights[~mask] = 0.0
W_blob = weights[mask]

# =====================================================================
# 3. CURVATURE-MATCHING SPECIFICATION OF TARGET CDF (f1)
# =====================================================================
# Sort the unselected data to analyze its trailing edge curvature
M_below_T = np.sort(M[M <= T])
prob_at_T = len(M_below_T) / M.size

# --- ESTIMATE LOCAL CURVATURE SLOPE ---
# Look at the upper 10% region of the unselected data right before threshold T
idx_near_T = int(len(M_below_T) * 0.90)
x_ref = M_below_T[idx_near_T]
p_ref = (idx_near_T / len(M_below_T)) * prob_at_T

# Compute the local linear derivative to match trailing slope direction
local_slope = (prob_at_T - p_ref) / (T - x_ref + 1e-12)

# Define absolute target window boundaries
min_x_above = T
max_x_above = T * 3.0

# Generate target intensity tracking coordinate steps
x_grid_above = np.linspace(min_x_above, max_x_above, 2000)

# --- CURVATURE FIT EXTRAPOLATION ---
# Construct an exponential decay asymptotic curve that launches with the 
# exact empirical slope found at T, but smoothly bends to terminate at 1.0 at 3T.
headroom = 1.0 - prob_at_T
# Calculate the matching decay rate parameter k to fuse the boundaries smoothly
k = local_slope / (headroom + 1e-12)

# Calculate the smooth continuation profile
raw_decay_curve = 1.0 - np.exp(-k * (x_grid_above - min_x_above))
# Normalize the tail explicitly so it spans exactly from prob_at_T up to 1.0 at 3T
f1_probabilities_above = prob_at_T + headroom * (raw_decay_curve / (raw_decay_curve[-1] + 1e-12))

# Combine below and above targets to seal the unified continuous f1 map
f1_x_below = M_below_T
f1_y_below = np.linspace(0.0, prob_at_T, len(M_below_T))

f1_x_curve = np.concatenate([f1_x_below, x_grid_above])
f1_y_curve = np.concatenate([f1_y_below, f1_probabilities_above])

# --- QUANTILE MATCHING OPERATION ---
sorted_all_M = np.sort(M.flatten())
ranks_in_f0 = np.searchsorted(sorted_all_M, M_blob_orig) / len(sorted_all_M)

# Execute numerical interpolation mapping through the custom curvature curve
M_quantile_matched_blob = np.interp(ranks_in_f0, f1_y_curve, f1_x_curve)

# =====================================================================
# 4. RECONSTRUCTION WITH ABSOLUTE BOUNDARY SAFEGUARD
# =====================================================================
M_final = M.copy()

# 1. Smoothly blend the mapped quantile values at the spatial boundaries
M_final[mask] = (1.0 - W_blob) * M[mask] + W_blob * M_quantile_matched_blob

# 2. Hard limit constraint check following alpha-blend step execution
M_final[mask] = np.clip(M_final[mask], None, max_x_above)

M_difference = M - M_final

# =====================================================================
# 5. VISUALIZATION (SPECTROGRAMS, PSDS, AND CONTINUOUS VIEW COHORTS)
# =====================================================================
fig, axes = plt.subplots(6, 1, figsize=(12, 28))

# Panel 1: Original Spectrogram
im0 = axes[0].pcolormesh(M, shading='auto', cmap='jet')
axes[0].set_title(f"Original Linear Spectrogram Matrix M (True Max: {M.max():.2e})")
axes[0].set_ylabel("Frequency Bins")
fig.colorbar(im0, ax=axes[0], label="Linear Power")

# Panel 2: Quantile-Matched Spectrogram
im1 = axes[1].pcolormesh(M_final, shading='auto', cmap='jet', vmin=M.min(), vmax=M.max())
axes[1].set_title(f"Quantile-Matched Spectrogram (Guaranteed Max Ceiling: {M_final[mask].max():.2e})")
axes[1].set_ylabel("Frequency Bins")
fig.colorbar(im1, ax=axes[1], label="Linear Power")

# Panel 3: 2D Spectrogram Difference Map
im2 = axes[2].pcolormesh(M_difference, shading='auto', cmap='jet')
axes[2].set_title("2D Spectrogram Difference Map (Energy Removed Via Quantile Specification)")
axes[2].set_ylabel("Frequency Bins")
fig.colorbar(im2, ax=axes[2], label="Amplitude Reduced")

# Panel 4: Linear PSD Curve Comparison
axes[3].plot(f_M, original_psd, label='PSD Original Spectrogram', color='red', alpha=0.6)
axes[3].plot(f_M, reference_psd, label='Whittaker Target Baseline', color='black', linestyle='--')
axes[3].plot(f_M, np.mean(M_final, axis=1), label='PSD Cleaned Spectrogram', color='green', linewidth=2)
axes[3].set_title("PSD Profile Verification Analysis")
axes[3].set_ylabel("Mean Linear Intensity")
axes[3].legend()
axes[3].grid(True)

# Panel 5: GLOBAL INTENSITY CDF VISUALIZATION
sorted_M_orig = np.sort(M.flatten())
f0_y = np.linspace(0.0, 1.0, len(sorted_M_orig))

sorted_M_final = np.sort(M_final.flatten())
f_final_y = np.linspace(0.0, 1.0, len(sorted_M_final))

axes[4].plot(sorted_M_orig, f0_y, label='Original Global CDF (f0)', color='red', alpha=0.5, linewidth=3)
axes[4].plot(f1_x_curve, f1_y_curve, label='Curvature-Matched Target CDF (f1)', color='blue', linestyle='--', linewidth=2)
axes[4].plot(sorted_M_final, f_final_y, label='Actual Resulting Global CDF', color='green', linewidth=2)
axes[4].axvline(T, color='purple', linestyle=':', label=f'Threshold T ({T:.2e})')
axes[4].axvline(max_x_above, color='cyan', linestyle=':', label=f'Hard Ceiling Upper Limit 3T ({max_x_above:.2e})')
axes[4].set_title("Global Intensity Cumulative Distribution Function (CDF) Specification")
axes[4].set_ylabel("Probability")
axes[4].set_xscale('log')
axes[4].legend()
axes[4].grid(True, which="both", ls="-")

# Panel 6: SUB-POPULATION COHORT CDF (Selected vs. Unselected Pixels)
unselected_pixels = np.sort(M[~mask])
selected_pixels_orig = np.sort(M_blob_orig)
selected_pixels_final = np.sort(M_final[mask])

y_unsel = np.linspace(0.0, 1.0, len(unselected_pixels))
y_sel_orig = np.linspace(0.0, 1.0, len(selected_pixels_orig))
y_sel_final = np.linspace(0.0, 1.0, len(selected_pixels_final))

axes[5].plot(unselected_pixels, y_unsel, label=f'Unselected Pixels (M <= T)', color='gray', linestyle='-', linewidth=2)
axes[5].plot(selected_pixels_orig, y_sel_orig, label=f'Selected Mask Pixels (Original Tail)', color='darkred', alpha=0.6, linewidth=2.5)
axes[5].plot(selected_pixels_final, y_sel_final, label=f'Selected Mask Pixels (Transformed Curvature Tail)', color='darkgreen', linewidth=2.5)

axes[5].axvline(T, color='purple', linestyle=':', label=f'Threshold T ({T:.2e})')
axes[5].axvline(max_x_above, color='cyan', linestyle=':', label=f'Max 3T Boundary ({max_x_above:.2e})')
axes[5].set_title("Isolated Sub-Population Empirical CDF Comparison")
axes[5].set_xlabel("Pixel Amplitude Value (Log Scale)")
axes[5].set_ylabel("Internal Cohort Probability")
axes[5].set_xscale('log')
axes[5].legend()
axes[5].grid(True, which="both", ls="-")

plt.tight_layout()
plt.show()

C:\Users\holcman\AppData\Local\Temp\ipykernel_9092\3484907175.py:46: RuntimeWarning: divide by zero encountered in divide
  weights = 1.0 - np.exp(-(distance_map / transition_width) ** 2)
C:\Users\holcman\AppData\Local\Temp\ipykernel_9092\3484907175.py:46: RuntimeWarning: invalid value encountered in divide
  weights = 1.0 - np.exp(-(distance_map / transition_width) ** 2)
